In [ ]:
from pathlib import Path
import sys

# Localizar la raiz del repositorio desde Jupyter o terminal
RAIZ_REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks").is_dir() and (p / "datos").is_dir()),
    None
)
if RAIZ_REPO is None:
    raise FileNotFoundError("Ejecuta desde el repositorio caso-itam")

sys.path.insert(0, str(RAIZ_REPO / "src"))


## Capa de integración 

In [ ]:
# Rutas, catálogo de alcaldías y carga de los parquet

from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- Rutas: las mismas del cuaderno 02-03 ---------------------
# Aquí solo LEEMOS lo que el cuaderno anterior guardó.
CARPETA_SALIDA = RAIZ_REPO / "datos" / "procesados"
assert CARPETA_SALIDA.exists(), f"No encuentro {CARPETA_SALIDA}"

# --- Traído del cuaderno 02-03 (celda 12): catálogo fijo ------
# Claves de municipio INEGI de la CDMX. La entidad es "09", así que
# la clave geográfica completa (CVEGEO) es "09" + clave, p. ej. "09002".
NOMBRE_ALC = pd.Series({
    "002": "Azcapotzalco",        "003": "Coyoacán",            "004": "Cuajimalpa de Morelos",
    "005": "Gustavo A. Madero",   "006": "Iztacalco",           "007": "Iztapalapa",
    "008": "La Magdalena Contreras", "009": "Milpa Alta",       "010": "Álvaro Obregón",
    "011": "Tláhuac",             "012": "Tlalpan",             "013": "Xochimilco",
    "014": "Benito Juárez",       "015": "Cuauhtémoc",          "016": "Miguel Hidalgo",
    "017": "Venustiano Carranza",
})

# --- Traído del cuaderno 02-03 (celda 28) ---------------------
def sin_acentos(texto):
    """'Álvaro Obregón' → 'ALVARO OBREGON'. Separa cada letra de su acento y descarta el acento."""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().upper()

# --- Traducción de años: LA trampa del día --------------------
# La ENVIPE de un año pregunta por delitos del año anterior.
# estimaciones_directas['anio'] es el año de EDICIÓN (2021–2025);
# el maestro se indexa por año de REFERENCIA (2020–2024).
# Toda conversión pasa por este diccionario, nunca por "anio - 1" suelto.
REFERENCIA_DE_EDICION = {2021: 2020, 2022: 2021, 2023: 2022, 2024: 2023, 2025: 2024}
ANIOS_REF = list(REFERENCIA_DE_EDICION.values())

# --- Carga -----------------------------------------------------
incidentes = pd.read_parquet(CARPETA_SALIDA / "envipe_incidentes.parquet")
directas   = pd.read_parquet(CARPETA_SALIDA / "estimaciones_directas.parquet")
carpetas   = pd.read_parquet(CARPETA_SALIDA / "fgj_carpetas.parquet")
corresp    = pd.read_csv(CARPETA_SALIDA / "correspondencia_bpcod_fgj_borrador_v2.csv",
                         encoding="utf-8-sig", dtype=str)

# --- Controles contra el día 3 ---------------------------------
esperado = {"incidentes": (incidentes, 186_329),
            "directas":   (directas,   320),
            "carpetas":   (carpetas,   2_098_743),
            "corresp":    (corresp,    357)}
for nombre, (df, n) in esperado.items():
    estado = "OK" if len(df) == n else "NO CUADRA"
    print(f"{nombre:<11} filas = {len(df):>10,}   esperado = {n:>10,}   {estado}")
    assert len(df) == n, f"{nombre} no coincide con el día 3"

# El catálogo debe ser exactamente el conjunto de claves de cada fuente ENVIPE
assert set(NOMBRE_ALC.index) == set(incidentes["alc_ocu"].dropna()) \
       == set(incidentes["alc_res"].dropna()) == set(directas["alc"]), "Claves distintas entre fuentes"
assert set(directas["anio"]) == set(REFERENCIA_DE_EDICION), "directas['anio'] no son años de edición"
print("\nCatálogo de 16 alcaldías: coincide con incidentes (ocurrencia y residencia) y con directas.")
print("directas['anio'] confirmado como año de EDICIÓN:", sorted(directas["anio"].unique()))

incidentes  filas =    186,329   esperado =    186,329   OK
directas    filas =        320   esperado =        320   OK
carpetas    filas =  2,098,743   esperado =  2,098,743   OK
corresp     filas =        357   esperado =        357   OK

Catálogo de 16 alcaldías: coincide con incidentes (ocurrencia y residencia) y con directas.
directas['anio'] confirmado como año de EDICIÓN: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [ ]:
# Tabla de claves: una fila por alcaldía, el nombre tal como aparece en cada fuente

# --- 1. Base: el catálogo fijo ----------------------------------
# cvegeo = entidad + municipio, la clave que usan el censo y el marco
# geoestadístico. cve_mun = solo el municipio, la que usa la ENVIPE.
claves = pd.DataFrame({"cve_mun": NOMBRE_ALC.index, "nombre": NOMBRE_ALC.values})
claves["cvegeo"] = "09" + claves["cve_mun"]
claves["nombre_ascii"] = claves["nombre"].map(sin_acentos)   # versión para comparar textos
claves = claves[["cvegeo", "cve_mun", "nombre", "nombre_ascii"]]

# --- 2. ENVIPE: cómo escribió cada edición el nombre ------------
# Tomamos residencia (ENT_RES, MUN_RES, NOM_MUN_RES) porque es la única
# geografía que trae nombre; la de ocurrencia solo trae clave.
res = incidentes.loc[incidentes["ENT_RES"] == "09", ["anio_envipe", "MUN_RES", "NOM_MUN_RES"]]

# Para cada par (clave, forma de escribir el nombre), en qué ediciones aparece.
variantes = (res.groupby(["MUN_RES", "NOM_MUN_RES"])["anio_envipe"]
                .agg(lambda s: f"{s.min()}–{s.max()}" if s.nunique() > 1 else str(s.min()))
                .reset_index())
variantes["texto"] = variantes["NOM_MUN_RES"] + " (" + variantes["anio_envipe"] + ")"
# Juntamos todas las variantes de una alcaldía en una sola celda, separadas por " | "
nombre_envipe = variantes.groupby("MUN_RES")["texto"].agg(" | ".join).rename("nombre_envipe")

# Pegado POR CLAVE, con su control.
# validate="one_to_one" detiene todo si una clave aparece dos veces de algún lado.
n_antes = len(claves)
claves = claves.merge(nombre_envipe, left_on="cve_mun", right_index=True,
                      how="left", validate="one_to_one")
print(f"ENVIPE  → entraron {n_antes}, salieron {len(claves)}, "
      f"sin nombre ENVIPE: {claves['nombre_envipe'].isna().sum()}")
print(f"         claves CDMX en la ENVIPE que no están en el catálogo: "
      f"{sorted(set(res['MUN_RES']) - set(claves['cve_mun']))}")

# Control de contenido: sin acentos y en mayúsculas, cada variante debe ser
# idéntica al nombre del catálogo. Si no, la clave y el nombre no corresponden.
nombre_ascii_por_clave = claves.set_index("cve_mun")["nombre_ascii"]
assert (variantes["NOM_MUN_RES"].map(sin_acentos)
        == variantes["MUN_RES"].map(nombre_ascii_por_clave)).all(), "Algún nombre ENVIPE no corresponde a su clave"

# --- 3. Fiscalía: de nombre a clave -----------------------------
# La Fiscalía NO trae clave, así que aquí sí hay que pasar por el nombre,
# pero una sola vez y verificando: diccionario nombre_ascii → clave.
CLAVE_POR_NOMBRE_FGJ = dict(zip(claves["nombre_ascii"], claves["cve_mun"]))

valores_fgj = carpetas["alcaldia_hecho"].value_counts(dropna=False)   # dropna=False: cuenta también los vacíos
asignacion = pd.DataFrame({"carpetas": valores_fgj})
# isinstance(v, str) evita aplicar sin_acentos a un NaN (que no es texto)
asignacion["cve_mun"] = [CLAVE_POR_NOMBRE_FGJ.get(sin_acentos(v)) if isinstance(v, str) else None
                         for v in asignacion.index]
print("\nFiscalía: cada valor de alcaldia_hecho y la clave que recibe")
print(asignacion.to_string())

empatan = asignacion[asignacion["cve_mun"].notna()]
assert len(empatan) == 16 and empatan["cve_mun"].is_unique, "No empataron exactamente 16 alcaldías distintas"

nombre_fgj = pd.Series(empatan.index, index=empatan["cve_mun"], name="nombre_fgj")
claves = claves.merge(nombre_fgj, left_on="cve_mun", right_index=True,
                      how="left", validate="one_to_one")

# Control de filas: lo que empata + lo que no = el total del archivo
con_clave = empatan["carpetas"].sum()
sin_clave = asignacion.loc[asignacion["cve_mun"].isna(), "carpetas"].sum()
print(f"\nFiscalía → con clave {con_clave:,} + sin clave {sin_clave:,} = {con_clave + sin_clave:,} "
      f"(archivo: {len(carpetas):,})")
assert con_clave + sin_clave == len(carpetas)
print(f"           alcaldías sin nombre FGJ: {claves['nombre_fgj'].isna().sum()}")

# --- 4. Resultado -----------------------------------------------
print()
print(claves.to_string(index=False))

ENVIPE  → entraron 16, salieron 16, sin nombre ENVIPE: 0
         claves CDMX en la ENVIPE que no están en el catálogo: []

Fiscalía: cada valor de alcaldia_hecho y la clave que recibe
                        carpetas cve_mun
alcaldia_hecho                          
CUAUHTEMOC                318787     015
IZTAPALAPA                304367     007
GUSTAVO A. MADERO         209034     005
BENITO JUAREZ             164576     014
COYOACAN                  142074     003
ALVARO OBREGON            140019     010
MIGUEL HIDALGO            130574     016
TLALPAN                   125799     012
VENUSTIANO CARRANZA       118736     017
AZCAPOTZALCO               98994     002
IZTACALCO                  89879     006
XOCHIMILCO                 67289     013
TLAHUAC                    57691     011
LA MAGDALENA CONTRERAS     37189     008
CUAJIMALPA DE MORELOS      33282     004
NaN                        24896     NaN
MILPA ALTA                 17971     009
CDMX (indeterminada)       17586    

### Fiscalía

In [ ]:
# Fiscalía: clave, año del hecho y "no delito" (reglas del cuaderno 02-03, celdas 28 y 29)

# Trabajamos directo sobre `carpetas`: el parquet crudo sigue en disco
# por si hay que volver a empezar.

# --- 1. Clave de alcaldía (el diccionario de la celda 2) ---------
carpetas["cve_mun"] = carpetas["alcaldia_hecho"].map(
    lambda v: CLAVE_POR_NOMBRE_FGJ.get(sin_acentos(v)) if isinstance(v, str) else None)

# --- 2. Año del hecho: de texto ("2020.0") a número --------------
# errors="coerce" convierte lo ilegible en NaN en vez de detenerse.
# Usamos la columna anio_hecho porque en el día 3 coincidió al 100%
# con el año de fecha_hecho, y no necesita leer las fechas.
carpetas["anio_h"] = pd.to_numeric(carpetas["anio_hecho"], errors="coerce")
print(f"anio_hecho vacío o ilegible: {carpetas['anio_h'].isna().sum():,}")

# --- 3. Registros que no son delitos -----------------------------
# Hasta 2020 la columna competencia venía vacía, por eso se combinan
# las dos variables: basta con que cualquiera de las dos lo diga.
carpetas["no_delito"] = (carpetas["competencia"].isin(["HECHO NO DELICTIVO", "INCOMPETENCIA"])
                         | (carpetas["categoria_delito"] == "HECHO NO DELICTIVO"))
print(f"Marcados como no delito: {carpetas['no_delito'].sum():,}  (día 3: 87,912)")
assert carpetas["no_delito"].sum() == 87_912

# --- 4. Filtros, uno por uno, con su control ---------------------
n0 = len(carpetas)
en_periodo = carpetas["anio_h"].between(2020, 2024)       # between incluye los dos extremos
n1 = en_periodo.sum()
es_delito = en_periodo & ~carpetas["no_delito"]           # ~ significa "no"
n2 = es_delito.sum()
print(f"\nArchivo completo:                      {n0:>10,}")
print(f"  hecho en 2020–2024:                  {n1:>10,}  (salen {n0 - n1:,})")
print(f"  y además es delito:                  {n2:>10,}  (salen {n1 - n2:,})")
assert n2 == 1_091_385, "No coincide con el total 2020–2024 del día 3"

# --- 5. Carpetas por alcaldía y año del hecho (todavía TODOS los delitos)
fgj = carpetas[es_delito]
t_fgj = (fgj.assign(alc=fgj["cve_mun"].fillna("sin alcaldía"))
            .pivot_table(index="alc", columns="anio_h", values="delito",
                         aggfunc="size", fill_value=0))
t_fgj.columns = t_fgj.columns.astype(int)                 # 2020.0 → 2020

total_anio = t_fgj.sum()
sin_alc = t_fgj.loc["sin alcaldía"]
print("\nPor año del hecho:")
print(pd.DataFrame({"total": total_anio,
                    "sin alcaldía": sin_alc,
                    "% sin alcaldía": (100 * sin_alc / total_anio).round(2)}).T.to_string())
print(f"\nAlcaldías con carpetas: {(t_fgj.index != 'sin alcaldía').sum()} (deben ser 16)")

anio_hecho vacío o ilegible: 559
Marcados como no delito: 87,912  (día 3: 87,912)

Archivo completo:                       2,098,743
  hecho en 2020–2024:                   1,126,832  (salen 971,911)
  y además es delito:                   1,091,385  (salen 35,447)

Por año del hecho:
anio_h               2020       2021       2022       2023       2024
total           201739.00  221641.00  230958.00  229704.00  207343.00
sin alcaldía      1224.00     703.00    1025.00    7144.00    7692.00
% sin alcaldía       0.61       0.32       0.44       3.11       3.71

Alcaldías con carpetas: 16 (deben ser 16)


In [ ]:
# Correspondencia v3: decisiones finales y versiones de C

# --- 0. Controles de entrada -------------------------------------
corresp["n"] = corresp["carpetas_2020_2024"].astype(int)
assert corresp["n"].sum() == 1_091_385, "El borrador no suma el total 2020–2024 de la celda 3"
assert corresp["delito"].is_unique, "Hay delitos repetidos: el pegado por delito duplicaría carpetas"

# --- 1. Nivel 1: delitos ENVIPE del mapa principal ---------------
NUCLEO = ["04", "05", "10", "11", "13", "14"]

# --- 2. Nivel 2: los dudosos que caen dentro del núcleo ----------
# Formato: delito FGJ → (BPCOD final, justificación). "—" = fuera.
DECISIONES = {
    "ROBO A PASAJERO / CONDUCTOR DE VEHICULO CON VIOLENCIA":
        ("05", "Robo a la persona en su vehículo; si se llevaran el vehículo sería 01"),
    "ROBO A TRANSEUNTE CONDUCTOR DE TAXI PUBLICO Y PRIVADO CON VIOLENCIA":
        ("05", "Robo a persona en transporte"),
    "ROBO A PASAJERO / CONDUCTOR DE TAXI CON VIOLENCIA":
        ("05", "Robo a persona en transporte"),
    "ROBO A REPARTIDOR CON VIOLENCIA":
        ("05", "Robo a persona en la calle; parte del botín puede ser de la empresa"),
    "ROBO A REPARTIDOR SIN VIOLENCIA":
        ("05", "Robo a persona en la calle; parte del botín puede ser de la empresa"),
    "ROBO A REPARTIDOR Y VEHICULO CON VIOLENCIA":
        ("05", "Robo a persona en la calle; parte del botín puede ser de la empresa"),
    "ROBO A REPARTIDOR Y VEHICULO SIN VIOLENCIA":
        ("05", "Robo a persona en la calle; parte del botín puede ser de la empresa"),
    "CONTRA LA INTIMIDAD SEXUAL":
        ("—",  "Difusión de contenido íntimo: no es hostigamiento, manoseo, exhibicionismo ni intento de violación"),
    "TENTATIVA DE VIOLACION":
        ("13", "La ENVIPE clasifica el intento de violación en 13, no en 14"),
}
# Control: cada nombre del diccionario debe existir tal cual en el borrador.
# Un error de dedo haría que la decisión no se aplicara sin avisar.
faltan = set(DECISIONES) - set(corresp["delito"])
assert not faltan, f"Delitos de DECISIONES que no están en el borrador: {faltan}"

# --- 3. Llenar decision_final y justificacion --------------------
# Por defecto, la decisión final es la propuesta del borrador.
corresp["decision_final"] = corresp["bpcod_propuesto"]
corresp["justificacion"] = np.where(corresp["confianza"] == "alta",
                                    "Confianza alta: se mantiene la propuesta", "")
for delito, (codigo, por_que) in DECISIONES.items():
    fila = corresp["delito"] == delito
    corresp.loc[fila, "decision_final"] = codigo
    corresp.loc[fila, "justificacion"] = por_que

# Los dudosos que no tocamos: si quedaron DENTRO del núcleo es un hueco
# (falta decidirlos); si quedaron fuera, solo cuentan en la versión amplia.
dudosa_sin = (corresp["confianza"] == "dudosa") & (corresp["justificacion"] == "")
huecos = corresp[dudosa_sin & corresp["decision_final"].isin(NUCLEO)]
assert huecos.empty, f"Dudosos dentro del núcleo sin decisión:\n{huecos[['delito', 'n']]}"
corresp.loc[dudosa_sin, "justificacion"] = "Fuera del núcleo: solo entra en la versión amplia"
assert (corresp["justificacion"] != "").all()

# --- 4. Una columna booleana por versión de C --------------------
corresp["v_nucleo"]      = corresp["decision_final"].isin(NUCLEO)
corresp["v_nucleo_alta"] = (corresp["v_nucleo"] & (corresp["confianza"] == "alta")
                            & (corresp["decision_final"] == corresp["bpcod_propuesto"]))
corresp["v_vf"]          = corresp["delito"] == "VIOLENCIA FAMILIAR"
corresp["v_amplio"]      = ~corresp["decision_final"].isin(["—", "03", "15"])

print("Carpetas 2020–2024 por BPCOD final:")
print(corresp.groupby("decision_final")["n"].sum().to_string())
print("\nVersiones de C (carpetas 2020–2024 · delitos FGJ que incluye):")
for v in ["v_nucleo_alta", "v_nucleo", "v_vf", "v_amplio"]:
    print(f"  {v:<14} {corresp.loc[corresp[v], 'n'].sum():>9,}  · {corresp[v].sum():>3} delitos")
print(f"  {'nucleo + vf':<14} {corresp.loc[corresp['v_nucleo'] | corresp['v_vf'], 'n'].sum():>9,}")

# --- 5. Guardar la v3 --------------------------------------------
RUTA_V3 = CARPETA_SALIDA / "correspondencia_bpcod_fgj_v3.csv"
corresp.drop(columns="n").to_csv(RUTA_V3, index=False, encoding="utf-8-sig")
print(f"\nGuardado: {RUTA_V3.name}")

Carpetas 2020–2024 por BPCOD final:
decision_final
01        36278
02        83715
03        25894
04        17696
05        96502
06        72274
07        32667
07/08     94374
08        20858
09        20877
10        89484
11        25263
12         2847
13        26273
14        10080
15            2
—        436301

Versiones de C (carpetas 2020–2024 · delitos FGJ que incluye):
  v_nucleo_alta    248,854  ·  63 delitos
  v_nucleo         265,298  ·  71 delitos
  v_vf             173,144  ·   1 delitos
  v_amplio         629,188  · 112 delitos
  nucleo + vf      438,442

Guardado: correspondencia_bpcod_fgj_v3.csv


In [ ]:
# C por alcaldía × año del hecho, cuatro versiones, y los factores de ajuste (sin alcaldía y corte)

# --- 1. Pegar las versiones a cada carpeta (por delito) ---------
VERSIONES = ["v_nucleo", "v_nucleo_alta", "v_vf", "v_amplio"]
n_antes = len(fgj)
fgj5 = fgj[["delito", "cve_mun", "anio_h", "fecha_inicio"]].merge(
    corresp[["delito"] + VERSIONES], on="delito",
    how="left", validate="many_to_one", indicator=True)
# indicator=True agrega la columna _merge: "both" si el delito sí estaba en la correspondencia
sin_empatar = (fgj5["_merge"] != "both").sum()
print(f"Pegado carpetas ↔ correspondencia: entraron {n_antes:,}, salieron {len(fgj5):,}, "
      f"sin empatar {sin_empatar:,}")
assert len(fgj5) == n_antes and sin_empatar == 0

fgj5["anio_h"] = fgj5["anio_h"].astype(int)
fgj5["v_nucleo_vf"] = fgj5["v_nucleo"] | fgj5["v_vf"]     # | significa "o"
VERS = ["v_nucleo", "v_nucleo_alta", "v_nucleo_vf", "v_amplio"]
con_alc = fgj5["cve_mun"].notna()

# --- 2. Conteos por alcaldía × año, con rejilla completa ---------
# La "rejilla" son las 80 combinaciones que DEBEN existir. Si alguna
# alcaldía no tuviera carpetas en un año, groupby no la crearía;
# reindex la crea y fillna(0) le pone cero en vez de perderla.
rejilla = pd.MultiIndex.from_product([claves["cve_mun"], ANIOS_REF], names=["cve_mun", "anio_ref"])
C = pd.concat(
    [fgj5[fgj5[v] & con_alc].groupby(["cve_mun", "anio_h"]).size().rename("C_" + v[2:])
     for v in VERS], axis=1)
C.index.names = ["cve_mun", "anio_ref"]   # año del hecho = año de referencia de la ENVIPE
print(f"\nCombinaciones con carpetas: {len(C)} · combinaciones vacías rellenadas: {len(rejilla) - len(C)}")
C = C.reindex(rejilla).fillna(0).astype(int)
assert len(C) == 80

# --- 3. Sin alcaldía: cuánto pesa por versión y año --------------
# Control: con alcaldía + sin alcaldía = total de la versión.
filas = []
for v in VERS:
    total = fgj5[v].groupby(fgj5["anio_h"]).sum()
    sin   = (fgj5[v] & ~con_alc).groupby(fgj5["anio_h"]).sum()
    con   = C["C_" + v[2:]].groupby(level="anio_ref").sum()
    assert (con + sin == total).all(), f"{v}: no cuadra con + sin = total"
    filas.append(pd.DataFrame({"version": v[2:], "anio_ref": total.index, "total": total.values,
                               "sin_alcaldia": sin.values, "pct_sin_alc": (100 * sin / total).values,
                               # Prorrateo: si las sin alcaldía se repartieran como las demás
                               "factor_sin_alc": (total / con).values}))
ajustes = pd.concat(filas, ignore_index=True)

# --- 4. Corte del archivo (31-ene-2025): rezago esperado en 2024 --
# Para 2020–2023 medimos qué fracción de las carpetas de cada año se
# abrió más de 31 días después de que terminó el año. Esa fracción es
# lo que le falta a 2024. str[:10] toma "aaaa-mm-dd" aunque la fecha
# traiga hora (los dos formatos que encontraste en el día 3).
f_ini = pd.to_datetime(fgj5["fecha_inicio"].str[:10], format="%Y-%m-%d", errors="coerce")
print(f"fecha_inicio ilegible: {f_ini.isna().sum()}")
limite = pd.to_datetime(fgj5["anio_h"].astype(str) + "-12-31") + pd.Timedelta(days=31)
fgj5["tardia"] = f_ini > limite

ajustes["pct_tardias"] = np.nan
ajustes["factor_corte"] = 1.0
for v in VERS:
    previas = fgj5[fgj5[v] & fgj5["anio_h"].between(2020, 2023)]
    frac = previas.groupby("anio_h")["tardia"].mean()
    fila = ajustes["version"] == v[2:]
    ajustes.loc[fila & ajustes["anio_ref"].le(2023), "pct_tardias"] = 100 * frac.values
    ajustes.loc[fila & (ajustes["anio_ref"] == 2024), "factor_corte"] = 1 / (1 - frac.mean())

print("\nAjustes por versión y año:")
print(ajustes.round({"pct_sin_alc": 2, "factor_sin_alc": 4, "pct_tardias": 2, "factor_corte": 4})
             .to_string(index=False))
print("\nPrimeras filas de C:")
print(C.head(10))

Pegado carpetas ↔ correspondencia: entraron 1,091,385, salieron 1,091,385, sin empatar 0

Combinaciones con carpetas: 80 · combinaciones vacías rellenadas: 0
fecha_inicio ilegible: 3

Ajustes por versión y año:
    version  anio_ref  total  sin_alcaldia  pct_sin_alc  factor_sin_alc  pct_tardias  factor_corte
     nucleo      2020  49981           403         0.81          1.0081         3.25        1.0000
     nucleo      2021  54445           328         0.60          1.0061         2.58        1.0000
     nucleo      2022  55803           385         0.69          1.0069         2.62        1.0000
     nucleo      2023  53813          2035         3.78          1.0393         2.28        1.0000
     nucleo      2024  51256          2192         4.28          1.0447          NaN        1.0275
nucleo_alta      2020  44764           356         0.80          1.0080         3.52        1.0000
nucleo_alta      2021  50065           318         0.64          1.0064         2.72        1.00

- El salto de “sin alcaldía” en 2023 es un cambio de registro, no del delito. En el núcleo pasa de 0.7% a 3.8%, así que una parte de la caída de carpetas con alcaldía en 2023–2024 es artificial. Como afecta a la ciudad entera y no a una alcaldía en particular, en el Rao-Yu conviene incluir efectos fijos de año: absorben cualquier artefacto común a las 16 alcaldías. Es otra razón, además de la que ya tenías sobre las covariables fijas del censo.
- Las carpetas tardías van bajando (3.25% → 2.28% en el núcleo). El factor promedio de 1.0275 puede sobreajustar un poco 2024. Si usaras solo 2023 saldría 1.023. La diferencia es de medio punto; con documentarlo basta.

In [ ]:
# Guardar resultados intermedios

# Tabla de claves (entregable 2). Las columnas de censo, CONAPO y SESNSP
# se agregan cuando tengamos esos archivos.
claves.to_csv(CARPETA_SALIDA / "claves_alcaldias.csv", index=False, encoding="utf-8-sig")

# C por alcaldía × año, cuatro versiones, conteos CRUDOS (sin ajustes).
# reset_index convierte cve_mun y anio_ref de índice a columnas normales,
# que es como deben ir en un parquet que otros cuadernos van a leer.
C.reset_index().to_parquet(CARPETA_SALIDA / "C_alcaldia_anio.parquet", index=False)

# Factores de ajuste por versión × año (la ciudad completa)
ajustes.to_csv(CARPETA_SALIDA / "ajustes_C.csv", index=False, encoding="utf-8-sig")

# Control: releer lo guardado y comparar contra lo que tenemos en memoria
releido = pd.read_parquet(CARPETA_SALIDA / "C_alcaldia_anio.parquet")
assert releido.set_index(["cve_mun", "anio_ref"]).equals(C), "Lo guardado no es igual a lo que hay en memoria"
print(f"claves_alcaldias.csv     {len(claves)} filas")
print(f"C_alcaldia_anio.parquet  {len(releido)} filas · columnas: {list(releido.columns)}")
print(f"ajustes_C.csv            {len(ajustes)} filas")

claves_alcaldias.csv     16 filas
C_alcaldia_anio.parquet  80 filas · columnas: ['cve_mun', 'anio_ref', 'C_nucleo', 'C_nucleo_alta', 'C_nucleo_vf', 'C_amplio']
ajustes_C.csv            20 filas


## ITER 2020

In [ ]:
# Totales por alcaldía, denominadores y covariables censales

RUTA_ITER = RAIZ_REPO / "datos" / "crudos" / "ITER" / "ITER_09CSV20.csv"
assert RUTA_ITER.exists(), f"No encuentro {RUTA_ITER}"

# dtype=str: las claves "002" y "0000" conservan sus ceros.
# utf-8-sig: el archivo trae BOM (el carácter invisible al inicio).
iter_raw = pd.read_csv(RUTA_ITER, dtype=str, encoding="utf-8-sig")
print(f"ITER leído: {iter_raw.shape[0]} filas × {iter_raw.shape[1]} columnas")

# --- 1. Solo las filas de total por alcaldía ---------------------
# Según la página 2 del descriptor: MUN distinto de "000" y LOC == "0000".
es_total_mun = (iter_raw["MUN"] != "000") & (iter_raw["LOC"] == "0000")
iter_mun = iter_raw[es_total_mun].copy()
print(f"Filas de total municipal: {len(iter_mun)} (deben ser 16)")
assert len(iter_mun) == 16 and iter_mun["MUN"].is_unique

# --- 2. Columnas que usaremos ------------------------------------
# Denominadores y covariables directas
BASE = ["POBTOT", "POBFEM", "P_18YMAS", "P_18YMAS_F", "P_12YMAS", "POCUPADA"]
# Insumos para los ejes socioeconómicos (componentes principales, celda 8)
PARA_PCA = ["P_15YMAS", "P15YM_SE", "GRAPROES", "P18YM_PB", "PSINDER", "PEA", "PDESOCUP",
            "TOTHOG", "HOGJEF_F", "VIVPARH_CV", "VPH_PISOTI", "VPH_AUTOM", "VPH_PC",
            "VPH_INTER", "VPH_SINTIC", "PRO_OCUP_C"]
faltan = [c for c in BASE + PARA_PCA if c not in iter_mun.columns]
assert not faltan, f"Columnas que no están en el ITER: {faltan}"

# Los asteriscos (*) o N/D de confidencialidad no deberían aparecer en
# totales municipales; si aparecieran, to_numeric los volvería NaN y
# este control lo detectaría.
for c in BASE + PARA_PCA:
    iter_mun[c] = pd.to_numeric(iter_mun[c], errors="coerce")
huecos = iter_mun[BASE + PARA_PCA].isna().sum()
assert huecos.sum() == 0, f"Valores no numéricos:\n{huecos[huecos > 0]}"

# --- 3. Control de suma contra el total de la entidad -------------
total_ent = pd.to_numeric(iter_raw.loc[(iter_raw["MUN"] == "000") & (iter_raw["LOC"] == "0000"),
                                       "POBTOT"]).iloc[0]
print(f"Suma de las 16 alcaldías: {iter_mun['POBTOT'].sum():,} · total CDMX: {total_ent:,}")
assert iter_mun["POBTOT"].sum() == total_ent

# --- 4. Pegar a la tabla de claves POR CLAVE, y anotar el nombre censal
iter_mun = iter_mun.rename(columns={"MUN": "cve_mun", "NOM_MUN": "nombre_censo"})
n_antes = len(claves)
censo = claves[["cvegeo", "cve_mun", "nombre", "nombre_ascii"]].merge(
    iter_mun[["cve_mun", "nombre_censo"] + BASE + PARA_PCA],
    on="cve_mun", how="left", validate="one_to_one", indicator=True)
print(f"Pegado claves ↔ censo: entraron {n_antes}, salieron {len(censo)}, "
      f"sin empatar {(censo['_merge'] != 'both').sum()}")
assert (censo["_merge"] == "both").all()
# Control de contenido: misma alcaldía bajo la misma clave
assert (censo["nombre_censo"].map(sin_acentos) == censo["nombre_ascii"]).all(), \
       "Algún nombre del censo no corresponde a su clave"
censo = censo.drop(columns="_merge")

# Agregar el nombre censal a la tabla maestra de claves (entregable 2)
if "nombre_censo" not in claves.columns:
    claves = claves.merge(censo[["cve_mun", "nombre_censo"]], on="cve_mun", validate="one_to_one")

# --- 5. Covariables censales directas ----------------------------
# Proporción femenina ENTRE la población de 18+ (la que entrevista la ENVIPE)
censo["prop_fem_18"] = censo["P_18YMAS_F"] / censo["P_18YMAS"]
# Proporción ocupada: el ITER define "ocupada" sobre la población de 12+,
# así que el denominador correcto es P_12YMAS, no POBTOT.
censo["prop_ocupada_12"] = censo["POCUPADA"] / censo["P_12YMAS"]

print()
print(censo[["cve_mun", "nombre", "POBTOT", "P_18YMAS", "prop_fem_18", "prop_ocupada_12"]]
      .round({"prop_fem_18": 3, "prop_ocupada_12": 3}).to_string(index=False))

censo.to_parquet(CARPETA_SALIDA / "censo2020_alcaldias.parquet", index=False)
print("\nGuardado: censo2020_alcaldias.parquet")

ITER leído: 666 filas × 286 columnas
Filas de total municipal: 16 (deben ser 16)
Suma de las 16 alcaldías: 9,209,944 · total CDMX: 9,209,944
Pegado claves ↔ censo: entraron 16, salieron 16, sin empatar 0

cve_mun                 nombre  POBTOT  P_18YMAS  prop_fem_18  prop_ocupada_12
    002           Azcapotzalco  432205    345123        0.535            0.617
    003               Coyoacán  614447    500374        0.538            0.609
    004  Cuajimalpa de Morelos  217686    162395        0.532            0.646
    005      Gustavo A. Madero 1173351    911482        0.527            0.599
    006              Iztacalco  404695    319021        0.533            0.626
    007             Iztapalapa 1835486   1383459        0.524            0.622
    008 La Magdalena Contreras  247622    189047        0.531            0.613
    009             Milpa Alta  152685    108278        0.521            0.660
    010         Álvaro Obregón  759137    590251        0.533            0.633
    0

In [ ]:
# Idempotencia: si esta celda ya se corrió, quitamos lo que agregó antes
# (incluidas las versiones con sufijo _x/_y del pegado duplicado).
censo = censo.drop(columns=[c for c in censo.columns
                            if c.startswith(("eje_socio", "ind_"))])

In [ ]:
# Eje(s) socioeconómico(s) por componentes principales

# --- 1. Indicadores como proporciones (no conteos) ---------------
# Los conteos crecen con el tamaño de la alcaldía; las proporciones no.
# Cada denominador corresponde al universo del indicador según el descriptor.
viv = censo["VIVPARH_CV"]          # viviendas particulares habitadas con características
indicadores = pd.DataFrame({
    "p_sin_escolaridad": censo["P15YM_SE"] / censo["P_15YMAS"],
    "grado_prom_escol":  censo["GRAPROES"],                    # ya viene como promedio
    "p_posbasica":       censo["P18YM_PB"] / censo["P_18YMAS"],
    "p_sin_salud":       censo["PSINDER"] / censo["POBTOT"],
    "tasa_desocupacion": censo["PDESOCUP"] / censo["PEA"],
    "p_jefatura_fem":    censo["HOGJEF_F"] / censo["TOTHOG"],
    "p_piso_tierra":     censo["VPH_PISOTI"] / viv,
    "p_automovil":       censo["VPH_AUTOM"] / viv,
    "p_computadora":     censo["VPH_PC"] / viv,
    "p_internet":        censo["VPH_INTER"] / viv,
    "p_sin_tic":         censo["VPH_SINTIC"] / viv,
    "ocupantes_cuarto":  censo["PRO_OCUP_C"],                  # ya viene como promedio
})
indicadores.index = censo["cve_mun"]
assert indicadores.notna().all().all() and np.isfinite(indicadores.values).all()

# --- 2. Estandarizar: media 0, desviación 1 ----------------------
# Sin esto, el grado de escolaridad (≈11) pesaría muchísimo más que una
# proporción (≈0.01) solo por su escala.
Z = (indicadores - indicadores.mean()) / indicadores.std(ddof=0)

# --- 3. PCA con álgebra lineal (descomposición en valores singulares)
# Z = U · diag(s) · Vt. Las filas de Vt son las direcciones (cargas) de
# cada componente; s² es proporcional a la varianza que explica cada uno.
U, s, Vt = np.linalg.svd(Z.values, full_matrices=False)
var_explicada = s**2 / (s**2).sum()

# El signo de un componente es arbitrario (la SVD puede devolverlo volteado).
# Lo fijamos para que se lea igual siempre: eje 1 positivo = MÁS rezago
# (correlación positiva con la proporción sin escolaridad).
#for j in range(2):
#    if np.corrcoef(Z.values @ Vt[j], indicadores["p_sin_escolaridad"])[0, 1] < 0:
#        Vt[j] = -Vt[j]
# Ancla del signo: la variable que más carga en cada eje.
# Eje 1: p_sin_escolaridad (positivo = más rezago).
# Eje 2: tasa_desocupacion (positivo = más desocupación).
ANCLAS = ["p_sin_escolaridad", "tasa_desocupacion"]
for j, ancla in enumerate(ANCLAS):
    if np.corrcoef(Z.values @ Vt[j], indicadores[ancla])[0, 1] < 0:
        Vt[j] = -Vt[j]


cargas = pd.DataFrame(Vt[:2].T, index=indicadores.columns, columns=["eje1", "eje2"])
puntajes = pd.DataFrame(Z.values @ Vt[:2].T, index=indicadores.index,
                        columns=["eje_socio_1", "eje_socio_2"])

print("Varianza explicada por componente:")
for j in range(4):
    print(f"  componente {j + 1}: {100 * var_explicada[j]:5.1f}%   (acumulada {100 * var_explicada[:j + 1].sum():5.1f}%)")
print("\nCargas (cuánto pesa cada indicador en cada eje):")
print(cargas.round(2).to_string())
print("\nPuntajes por alcaldía (eje 1 positivo = más rezago):")
print(puntajes.assign(nombre=censo.set_index("cve_mun")["nombre"])
              .sort_values("eje_socio_1").round(2).to_string())

# --- 4. Agregar a la tabla censal y guardar ------------------------
censo = censo.merge(puntajes, left_on="cve_mun", right_index=True, validate="one_to_one")
censo = censo.merge(indicadores.add_prefix("ind_"), left_on="cve_mun", right_index=True,
                    validate="one_to_one")
censo.to_parquet(CARPETA_SALIDA / "censo2020_alcaldias.parquet", index=False)
cargas.to_csv(CARPETA_SALIDA / "pca_cargas.csv", encoding="utf-8-sig")
print("\nActualizado: censo2020_alcaldias.parquet · guardado pca_cargas.csv")

Varianza explicada por componente:
  componente 1:  74.3%   (acumulada  74.3%)
  componente 2:  12.3%   (acumulada  86.7%)
  componente 3:   5.7%   (acumulada  92.4%)
  componente 4:   3.0%   (acumulada  95.3%)

Cargas (cuánto pesa cada indicador en cada eje):
                   eje1  eje2
p_sin_escolaridad  0.30  0.05
grado_prom_escol  -0.32 -0.18
p_posbasica       -0.32 -0.10
p_sin_salud        0.26  0.30
tasa_desocupacion -0.11  0.64
p_jefatura_fem    -0.26  0.38
p_piso_tierra      0.28 -0.30
p_automovil       -0.24 -0.41
p_computadora     -0.33 -0.08
p_internet        -0.33  0.06
p_sin_tic          0.31 -0.18
ocupantes_cuarto   0.33  0.08

Puntajes por alcaldía (eje 1 positivo = más rezago):
         eje_socio_1  eje_socio_2                  nombre
cve_mun                                                  
014            -6.38        -1.83           Benito Juárez
016            -3.74        -1.56          Miguel Hidalgo
003            -2.73         0.66                Coyoacán
002  

## CONAPO 
### Población a mitad de año por municipio y grupos quinquenales de eddad (1990-2024)

In [ ]:
CARPETA_CONAPO = RAIZ_REPO / "datos" / "crudos" / "CONAPO"
archivos = sorted(CARPETA_CONAPO.glob("*.csv"))
print("CSV en la carpeta:", [a.name for a in archivos])
assert len(archivos) == 1, "Esperaba exactamente un CSV en Raw Data\\CONAPO"
RUTA_CONAPO = archivos[0]
print(f"Tamaño: {RUTA_CONAPO.stat().st_size / 1e6:,.1f} MB")

# Probamos codificaciones leyendo solo 5 filas (instantáneo aunque el archivo pese mucho)
for codif in ["utf-8-sig", "latin-1"]:
    try:
        muestra = pd.read_csv(RUTA_CONAPO, nrows=5, dtype=str, encoding=codif)
        CODIF_CONAPO = codif
        break
    except UnicodeDecodeError:
        continue
print(f"Codificación: {CODIF_CONAPO}\n")
print(muestra.T)          # .T voltea la tabla: una fila por columna, más fácil de leer

CSV en la carpeta: ['pobproy_quinq1.csv']
Tamaño: 36.7 MB
Codificación: utf-8-sig

                              0               1               2               3               4
CLAVE                      1001            1001            1001            1001            1001
CLAVE_ENT                     1               1               1               1               1
NOM_ENT          Aguascalientes  Aguascalientes  Aguascalientes  Aguascalientes  Aguascalientes
NOM_MUN          Aguascalientes  Aguascalientes  Aguascalientes  Aguascalientes  Aguascalientes
SEXO                    HOMBRES         MUJERES         HOMBRES         MUJERES         HOMBRES
ANO                        1990            1990            1991            1991            1992
POB_TOTAL                243891          255948          252446          264829          261132
POB_00_04                 35865           34812           36879           35825           37908
POB_05_09                 32966           31927      

In [ ]:
# población total y de 18+ por alcaldía y año

conapo_raw = pd.read_csv(RUTA_CONAPO, dtype=str, encoding=CODIF_CONAPO)
print(f"CONAPO completo: {len(conapo_raw):,} filas")
print("Valores de SEXO:", conapo_raw["SEXO"].value_counts().to_dict())

# --- 1. Clave: "9002" → "09002" → cve_mun "002" -----------------
conapo_raw["cvegeo"] = conapo_raw["CLAVE"].str.strip().str.zfill(5)
conapo_raw["cve_mun"] = conapo_raw["cvegeo"].str[2:]

# --- 2. Filtro: CDMX, 2020–2024, en dos pasos con su control -----
es_cdmx = conapo_raw["cvegeo"].str[:2] == "09"
conapo = conapo_raw[es_cdmx & conapo_raw["ANO"].astype(int).isin(ANIOS_REF)].copy()
print(f"CDMX: {es_cdmx.sum():,} filas · y además 2020–2024: {len(conapo)} filas "
      f"(esperadas 16 × 5 × 2 sexos = 160)")
print("Claves CDMX en CONAPO que no están en el catálogo:",
      sorted(set(conapo["cve_mun"]) - set(claves["cve_mun"])))
assert len(conapo) == 160 and set(conapo["cve_mun"]) == set(claves["cve_mun"])

# --- 3. A números, y control interno: la suma de edades = POB_TOTAL
EDADES = [c for c in conapo.columns if c.startswith("POB_") and c != "POB_TOTAL"]
print(f"Grupos de edad: {len(EDADES)} → {EDADES[0]} … {EDADES[-1]}")
for c in EDADES + ["POB_TOTAL", "ANO"]:
    conapo[c] = pd.to_numeric(conapo[c])
dif = (conapo[EDADES].sum(axis=1) - conapo["POB_TOTAL"]).abs().max()
print(f"Máxima diferencia entre suma de edades y POB_TOTAL: {dif}")
assert dif <= 1, "Los grupos de edad no suman el total"

# --- 4. Fracción de 18–19 dentro de 15–19, del censo, por alcaldía y sexo
# Del ITER: (P_15A19 − P_15A17) / P_15A19 = personas de 18 y 19 entre las de 15 a 19.
iter_1519 = iter_mun.set_index("cve_mun")
frac_1819 = {}
for sexo, suf in [("HOMBRES", "_M"), ("MUJERES", "_F")]:
    p1519 = pd.to_numeric(iter_1519["P_15A19" + suf])
    p1517 = pd.to_numeric(iter_1519["P_15A17" + suf])
    frac_1819[sexo] = (p1519 - p1517) / p1519
frac_1819 = pd.DataFrame(frac_1819)
print("\nFracción de 18–19 dentro de 15–19 (censo 2020):")
print(f"  rango hombres {frac_1819['HOMBRES'].min():.3f}–{frac_1819['HOMBRES'].max():.3f} · "
      f"mujeres {frac_1819['MUJERES'].min():.3f}–{frac_1819['MUJERES'].max():.3f}")

# Asignamos a cada fila de CONAPO la fracción de su alcaldía y su sexo
conapo["frac_1819"] = [frac_1819.loc[c, s] for c, s in zip(conapo["cve_mun"], conapo["SEXO"])]
DE_20_EN_ADELANTE = EDADES[EDADES.index("POB_20_24"):]
conapo["pob_18ymas"] = conapo["frac_1819"] * conapo["POB_015_019"] + conapo[DE_20_EN_ADELANTE].sum(axis=1)

# --- 5. Una fila por alcaldía × año ------------------------------
pob = (conapo.pivot_table(index=["cve_mun", "ANO"], columns="SEXO",
                          values=["POB_TOTAL", "pob_18ymas"], aggfunc="sum"))
pob = pd.DataFrame({
    "pob_total_conapo":   pob[("POB_TOTAL", "HOMBRES")] + pob[("POB_TOTAL", "MUJERES")],
    "pob_18_conapo":      pob[("pob_18ymas", "HOMBRES")] + pob[("pob_18ymas", "MUJERES")],
    "pob_18_fem_conapo":  pob[("pob_18ymas", "MUJERES")],
})
pob.index.names = ["cve_mun", "anio_ref"]
pob = pob.reindex(rejilla)                      # la misma rejilla de 80 de la celda 5
assert len(pob) == 80 and pob.notna().all().all()

# --- 6. Comparación con el censo 2020 ----------------------------
# No deben ser idénticas: el censo es al 15 de marzo y CONAPO a mitad de año,
# además de la conciliación demográfica. Lo que importa es que sean cercanas.
c20 = pob.xs(2020, level="anio_ref").join(censo.set_index("cve_mun")[["nombre", "POBTOT", "P_18YMAS"]])
c20["dif_total_%"] = 100 * (c20["pob_total_conapo"] / c20["POBTOT"] - 1)
c20["dif_18_%"]    = 100 * (c20["pob_18_conapo"] / c20["P_18YMAS"] - 1)
print("\nCONAPO 2020 contra censo 2020 (% de diferencia):")
print(c20[["nombre", "POBTOT", "pob_total_conapo", "dif_total_%", "dif_18_%"]]
      .round({"pob_total_conapo": 0, "dif_total_%": 2, "dif_18_%": 2}).to_string())

# --- 7. Nombre CONAPO en la tabla de claves (solo para documentar) --
nom_conapo = conapo.drop_duplicates("cve_mun").set_index("cve_mun")["NOM_MUN"].rename("nombre_conapo")
if "nombre_conapo" not in claves.columns:
    claves = claves.merge(nom_conapo, left_on="cve_mun", right_index=True, validate="one_to_one")
distintos = claves[claves["nombre_conapo"].map(sin_acentos) != claves["nombre_ascii"]]
print("\nNombres CONAPO que difieren del catálogo (no importa: pegamos por clave):")
print(distintos[["cve_mun", "nombre", "nombre_conapo"]].to_string(index=False) if len(distintos) else "  ninguno")

pob.reset_index().to_parquet(CARPETA_SALIDA / "poblacion_conapo_alcaldia_anio.parquet", index=False)
print("\nGuardado: poblacion_conapo_alcaldia_anio.parquet")

CONAPO completo: 252,450 filas
Valores de SEXO: {'HOMBRES': 126225, 'MUJERES': 126225}
CDMX: 1,632 filas · y además 2020–2024: 160 filas (esperadas 16 × 5 × 2 sexos = 160)
Claves CDMX en CONAPO que no están en el catálogo: []
Grupos de edad: 18 → POB_00_04 … POB_85_mm
Máxima diferencia entre suma de edades y POB_TOTAL: 0

Fracción de 18–19 dentro de 15–19 (censo 2020):
  rango hombres 0.398–0.439 · mujeres 0.400–0.429

CONAPO 2020 contra censo 2020 (% de diferencia):
                         nombre   POBTOT  pob_total_conapo  dif_total_%  dif_18_%
cve_mun                                                                          
002                Azcapotzalco   432205            439115         1.60      0.87
003                    Coyoacán   614447            621774         1.19      0.48
004       Cuajimalpa de Morelos   217686            221323         1.67      0.74
005           Gustavo A. Madero  1173351           1187205         1.18      0.02
006                   Iztacalco   40

Propongo 18 y más como denominador principal de las tasas, porque la ENVIPE entrevista a personas de 18 y más. El INEGI publica su tasa de incidencia por cada 100 mil habitantes de 18 y más, así que nuestra tasa queda en la misma escala. Las carpetas incluyen víctimas menores de edad, pero el núcleo ya excluyó las categorías explícitamente de menores.

In [ ]:
# maestro_alcaldia v0: C + población + covariables censales

# --- 1. C (80 filas) + población CONAPO (80 filas), por clave y año
m = C.reset_index().merge(pob.reset_index(), on=["cve_mun", "anio_ref"],
                          how="outer", validate="one_to_one", indicator=True)
print(f"C ↔ población: {len(m)} filas · sin empatar {(m['_merge'] != 'both').sum()}")
assert len(m) == 80 and (m["_merge"] == "both").all()
m = m.drop(columns="_merge")

# --- 2. + covariables censales (16 filas → se repiten en los 5 años)
# validate="many_to_one": muchas filas de m por cada alcaldía del censo.
COLS_CENSO = ["cve_mun", "cvegeo", "nombre", "POBTOT", "P_18YMAS",
              "prop_fem_18", "prop_ocupada_12", "eje_socio_1", "eje_socio_2"]
m = m.merge(censo[COLS_CENSO].rename(columns={"POBTOT": "pob_total_censo2020",
                                              "P_18YMAS": "pob_18_censo2020"}),
            on="cve_mun", how="left", validate="many_to_one", indicator=True)
print(f"+ censo: {len(m)} filas · sin empatar {(m['_merge'] != 'both').sum()}")
assert len(m) == 80 and (m["_merge"] == "both").all()
m = m.drop(columns="_merge")

# --- 3. Tasas por cada 100 mil: principal 18+, robustez población total
for v in ["nucleo", "nucleo_alta", "nucleo_vf", "amplio"]:
    m[f"tasa18_{v}"] = 1e5 * m[f"C_{v}"] / m["pob_18_conapo"]
    m[f"tasa_tot_{v}"] = 1e5 * m[f"C_{v}"] / m["pob_total_conapo"]

# --- 4. Factores de ajuste de la versión principal (iguales para las 16 por año)
aj = (ajustes.loc[ajustes["version"] == "nucleo", ["anio_ref", "factor_corte", "factor_sin_alc"]]
             .rename(columns={"factor_corte": "factor_corte_nucleo",
                              "factor_sin_alc": "factor_sin_alc_nucleo"}))
m = m.merge(aj, on="anio_ref", how="left", validate="many_to_one")
assert m[["factor_corte_nucleo", "factor_sin_alc_nucleo"]].notna().all().all()

# --- 5. Orden de columnas y guardado ----------------------------
primero = ["cvegeo", "cve_mun", "nombre", "anio_ref"]
m = m[primero + [c for c in m.columns if c not in primero]].sort_values(["cve_mun", "anio_ref"])
m = m.reset_index(drop=True)
assert not m.duplicated(["cve_mun", "anio_ref"]).any()

m.to_parquet(CARPETA_SALIDA / "maestro_alcaldia_v0.parquet", index=False)
print(f"\nGuardado: maestro_alcaldia_v0.parquet · {m.shape[0]} filas × {m.shape[1]} columnas")
print("Columnas:", list(m.columns))

# --- 6. Vistazo: tasa del núcleo por 100 mil de 18+ ---------------
print("\nCarpetas del núcleo por cada 100 mil habitantes de 18+ (CONAPO):")
print(m.pivot(index="nombre", columns="anio_ref", values="tasa18_nucleo")
       .round(0).sort_values(2022, ascending=False).to_string())

C ↔ población: 80 filas · sin empatar 0
+ censo: 80 filas · sin empatar 0

Guardado: maestro_alcaldia_v0.parquet · 80 filas × 27 columnas
Columnas: ['cvegeo', 'cve_mun', 'nombre', 'anio_ref', 'C_nucleo', 'C_nucleo_alta', 'C_nucleo_vf', 'C_amplio', 'pob_total_conapo', 'pob_18_conapo', 'pob_18_fem_conapo', 'pob_total_censo2020', 'pob_18_censo2020', 'prop_fem_18', 'prop_ocupada_12', 'eje_socio_1', 'eje_socio_2', 'tasa18_nucleo', 'tasa_tot_nucleo', 'tasa18_nucleo_alta', 'tasa_tot_nucleo_alta', 'tasa18_nucleo_vf', 'tasa_tot_nucleo_vf', 'tasa18_amplio', 'tasa_tot_amplio', 'factor_corte_nucleo', 'factor_sin_alc_nucleo']

Carpetas del núcleo por cada 100 mil habitantes de 18+ (CONAPO):
anio_ref                  2020    2021    2022    2023    2024
nombre                                                        
Cuauhtémoc              1305.0  1443.0  1610.0  1645.0  1561.0
Venustiano Carranza      901.0  1009.0   986.0   959.0   873.0
Miguel Hidalgo           931.0   963.0   918.0   844.0   817.

- Cuauhtémoc está muy arriba de todas, entre 1,305 y 1,645 carpetas por 100 mil, casi el doble que la siguiente. Era lo esperado por la población flotante: el numerador es por ocurrencia y el denominador por residencia.
- Azcapotzalco cae 25% de 2022 a 2023 (de 829 a 625) y no se recupera. Ninguna otra alcaldía cae tanto. Puede ser el cambio de registro de “sin alcaldía” concentrado en una fiscalía, o un cambio real. Lo anoto como algo a revisar.

## SESNSP 
### 2015 - 2025 (Fuero Común - Delitos). Incidencia delictiva municipal

In [ ]:
# ver qué trae antes de leerlo

CARPETA_SESNSP = RAIZ_REPO / "datos" / "crudos" / "SESNSP" / "Municipal-Delitos-2015-2025_ago2026"
RUTA_SESNSP = CARPETA_SESNSP / "Municipal-Delitos-2015-2025_ago2026.csv"
assert RUTA_SESNSP.exists(), f"No encuentro {RUTA_SESNSP}"
print(f"Tamaño: {RUTA_SESNSP.stat().st_size / 1e6:,.1f} MB")

# El SESNSP suele publicar en latin-1 o cp1252 (Windows); probamos en orden.
for codif in ["utf-8-sig", "latin-1", "cp1252"]:
    try:
        muestra = pd.read_csv(RUTA_SESNSP, nrows=5, dtype=str, encoding=codif)
        CODIF_SESNSP = codif
        break
    except UnicodeDecodeError:
        continue
print(f"Codificación: {CODIF_SESNSP} · columnas: {muestra.shape[1]}\n")
print(muestra.T)

Tamaño: 378.9 MB
Codificación: latin-1 · columnas: 21

                                                       0                                 1                                 2                                 3                                 4
Año                                                 2015                              2015                              2015                              2015                              2015
Clave_Ent                                              1                                 1                                 1                                 1                                 1
Entidad                                   Aguascalientes                    Aguascalientes                    Aguascalientes                    Aguascalientes                    Aguascalientes
Cve. Municipio                                      1001                              1001                              1001                              1001               

In [ ]:
# CDMX, 2020–2024, total anual por fila

MESES_COL = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio",
             "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
anios_txt = [str(a) for a in ANIOS_REF]

# --- 1. Lectura por bloques (chunksize) --------------------------
# read_csv con chunksize devuelve un "iterador": cada vuelta del for
# trae el siguiente bloque de 500 mil filas.
partes, filas_leidas = [], 0
for bloque in pd.read_csv(RUTA_SESNSP, dtype=str, encoding=CODIF_SESNSP, chunksize=500_000):
    filas_leidas += len(bloque)
    sel = (bloque["Clave_Ent"].str.strip() == "9") & bloque["Año"].str.strip().isin(anios_txt)
    partes.append(bloque[sel])
sesnsp = pd.concat(partes, ignore_index=True)
print(f"Filas leídas: {filas_leidas:,} · CDMX 2020–2024: {len(sesnsp):,}")

# --- 2. Clave de alcaldía: "9002" → "09002" → "002" ----------------
sesnsp["cvegeo"] = sesnsp["Cve. Municipio"].str.strip().str.zfill(5)
sesnsp["cve_mun"] = sesnsp["cvegeo"].str[2:]
fuera = sesnsp.loc[~sesnsp["cve_mun"].isin(claves["cve_mun"]), ["Cve. Municipio", "Municipio"]]
print("Claves que no son ninguna de las 16 alcaldías:")
print(fuera.value_counts().to_string() if len(fuera) else "  ninguna")

# --- 3. Total anual = suma de los 12 meses -------------------------
for mes in MESES_COL:
    sesnsp[mes] = pd.to_numeric(sesnsp[mes], errors="coerce")
print(f"Celdas de mes vacías o ilegibles: {sesnsp[MESES_COL].isna().sum().sum()}")
sesnsp["total_anio"] = sesnsp[MESES_COL].sum(axis=1)
sesnsp["anio_ref"] = sesnsp["Año"].astype(int)

# --- 4. Control de rejilla: mismas categorías en cada alcaldía y año
filas_por = sesnsp.groupby(["anio_ref", "cve_mun"]).size()
print(f"\nFilas por alcaldía-año: mín {filas_por.min()} · máx {filas_por.max()} "
      f"(si son iguales, es la rejilla completa de categorías)")

# --- 5. Totales y catálogo, para decidir qué categorías usar -------
print("\nTotal fuero común CDMX por año (las 16 alcaldías + lo que no sea alcaldía):")
print(sesnsp.groupby("anio_ref")["total_anio"].sum().to_string())

CAT = ["Bien jurídico afectado", "Tipo de delito", "Subtipo de delito"]
catalogo_sesnsp = (sesnsp.groupby(CAT)["total_anio"].sum()
                         .sort_values(ascending=False).rename("total_2020_2024"))
print(f"\nCatálogo: {len(catalogo_sesnsp)} combinaciones bien jurídico / tipo / subtipo")
pd.set_option("display.max_rows", 200)
print(catalogo_sesnsp.to_string())

sesnsp.to_parquet(CARPETA_SALIDA / "sesnsp_cdmx_2020_2024.parquet", index=False)
print("\nGuardado: sesnsp_cdmx_2020_2024.parquet")

Filas leídas: 2,562,994 · CDMX 2020–2024: 8,330
Claves que no son ninguna de las 16 alcaldías:
Cve. Municipio  Municipio      
9998            No Especificado    490
Celdas de mes vacías o ilegibles: 0

Filas por alcaldía-año: mín 98 · máx 98 (si son iguales, es la rejilla completa de categorías)

Total fuero común CDMX por año (las 16 alcaldías + lo que no sea alcaldía):
anio_ref
2020    198155
2021    223742
2022    232644
2023    234475
2024    222753

Catálogo: 55 combinaciones bien jurídico / tipo / subtipo
Bien jurídico afectado                              Tipo de delito                                                                 Subtipo de delito                                                            
La familia                                          Violencia familiar                                                             Violencia familiar                                                               173471
El patrimonio                                       Ro

- 8,330 filas = 17 claves × 98 categorías × 5 años. Las 17 claves son las 16 alcaldías más 9998, “No Especificado”. no hay celdas vacías.
- Los totales anuales del SESNSP se parecen a los de la Fiscalía, pero no son iguales. En 2024 el SESNSP da 222,753 y la Fiscalía, por año del hecho, 207,343. La diferencia es justo la de fecha. El SESNSP cuenta por registro, así que no sufre el rezago del corte de enero de 2025 que tiene nuestro conteo por hecho.

In [ ]:
# total y núcleo por alcaldía-año, al maestro

# --- 1. Núcleo SESNSP: subtipo → BPCOD -----------------------------
NUCLEO_SESNSP = {
    "Robo a casa habitación": "04",
    "Robo a transeúnte en vía pública": "05",
    "Robo a transeúnte en espacio abierto al público": "05",
    "Robo en transporte público colectivo": "05",
    "Robo en transporte público individual": "05",
    "Robo en transporte individual": "05",
    "Amenazas": "10",
    "Lesiones dolosas": "11",
    "Abuso sexual": "13", "Acoso sexual": "13", "Hostigamiento sexual": "13",
    "Violación simple": "14", "Violación equiparada": "14",
}
faltan = set(NUCLEO_SESNSP) - set(sesnsp["Subtipo de delito"].unique())
assert not faltan, f"Subtipos escritos distinto al archivo: {faltan}"
sesnsp["bpcod"] = sesnsp["Subtipo de delito"].map(NUCLEO_SESNSP)

# --- 2. Peso del "No Especificado" (9998) ------------------------
es_alc = sesnsp["cve_mun"].isin(claves["cve_mun"])
peso = (sesnsp.loc[~es_alc].groupby("anio_ref")["total_anio"].sum()
        / sesnsp.groupby("anio_ref")["total_anio"].sum() * 100)
print("% del total SESNSP sin alcaldía (9998), por año:", peso.round(2).to_dict())

# --- 3. Conteos por alcaldía × año (solo las 16) -------------------
alc = sesnsp[es_alc]
llave = ["cve_mun", "anio_ref"]
S = pd.concat([
    alc.groupby(llave)["total_anio"].sum().rename("S_total"),
    alc[alc["bpcod"].notna()].groupby(llave)["total_anio"].sum().rename("S_nucleo"),
    alc[alc["Subtipo de delito"] == "Violencia familiar"].groupby(llave)["total_anio"].sum().rename("S_vf"),
], axis=1).reindex(rejilla).fillna(0).astype(int)
assert len(S) == 80

print("\nNúcleo por año, ciudad (16 alcaldías): SESNSP por registro vs. Fiscalía por hecho")
comp = pd.DataFrame({"SESNSP": S["S_nucleo"].groupby(level="anio_ref").sum(),
                     "FGJ": C["C_nucleo"].groupby(level="anio_ref").sum()})
comp["SESNSP/FGJ"] = (comp["SESNSP"] / comp["FGJ"]).round(3)
print(comp.to_string())

# --- 4. Al maestro (idempotente) ----------------------------------
m = m.drop(columns=[c for c in m.columns if c.startswith(("S_", "tasa18_sesnsp", "z_sesnsp"))])
n_antes = len(m)
m = m.merge(S.reset_index(), on=llave, how="left", validate="one_to_one", indicator=True)
print(f"\nmaestro ↔ SESNSP: entraron {n_antes}, salieron {len(m)}, "
      f"sin empatar {(m['_merge'] != 'both').sum()}")
assert len(m) == 80 and (m["_merge"] == "both").all()
m = m.drop(columns="_merge")

# --- 5. Tasas y estandarización -----------------------------------
# Estandarizamos el LOGARITMO de la tasa sobre el panel completo (80 filas),
# no año por año: así la covariable conserva la variación temporal, que es
# justo lo que queremos de ella en el Rao-Yu.
for s in ["total", "nucleo"]:
    m[f"tasa18_sesnsp_{s}"] = 1e5 * m[f"S_{s}"] / m["pob_18_conapo"]
    lt = np.log(m[f"tasa18_sesnsp_{s}"])
    m[f"z_sesnsp_{s}"] = (lt - lt.mean()) / lt.std(ddof=0)

# --- 6. ¿Qué tan parecido es el SESNSP a nuestro C? (circularidad) --
lc, ls, lt = np.log(m["tasa18_nucleo"]), np.log(m["tasa18_sesnsp_nucleo"]), np.log(m["tasa18_sesnsp_total"])
print("\nCorrelación (log tasas, 80 alcaldía-año):")
print(f"  C núcleo FGJ  vs  SESNSP núcleo : {np.corrcoef(lc, ls)[0, 1]:.3f}")
print(f"  C núcleo FGJ  vs  SESNSP total  : {np.corrcoef(lc, lt)[0, 1]:.3f}")
print("  Spearman por año (núcleo vs núcleo):",
      {a: round(g["tasa18_nucleo"].corr(g["tasa18_sesnsp_nucleo"], method="spearman"), 3)
       for a, g in m.groupby("anio_ref")})

m.to_parquet(CARPETA_SALIDA / "maestro_alcaldia_v1.parquet", index=False)
print(f"\nGuardado: maestro_alcaldia_v1.parquet · {m.shape[0]} × {m.shape[1]}")

% del total SESNSP sin alcaldía (9998), por año: {2020: 0.02, 2021: 0.27, 2022: 0.25, 2023: 3.13, 2024: 3.55}

Núcleo por año, ciudad (16 alcaldías): SESNSP por registro vs. Fiscalía por hecho
          SESNSP    FGJ  SESNSP/FGJ
anio_ref                           
2020       47933  49578       0.967
2021       53685  54117       0.992
2022       55802  55418       1.007
2023       53208  51778       1.028
2024       51811  49064       1.056

maestro ↔ SESNSP: entraron 80, salieron 80, sin empatar 0

Correlación (log tasas, 80 alcaldía-año):
  C núcleo FGJ  vs  SESNSP núcleo : 0.993
  C núcleo FGJ  vs  SESNSP total  : 0.910
  Spearman por año (núcleo vs núcleo): {2020: np.float64(0.994), 2021: np.float64(0.994), 2022: np.float64(0.988), 2023: np.float64(0.994), 2024: np.float64(1.0)}

Guardado: maestro_alcaldia_v1.parquet · 80 × 34


1. El salto de “sin alcaldía” en 2023 ya está confirmado como cambio de registro. El SESNSP lo tiene igual: pasa de 0.25% a 3.13%. Como el SESNSP no pasa por nuestro procesamiento, el salto viene del origen, de la Fiscalía. 
2. La razón SESNSP/FGJ sube de 0.97 a 1.06 de forma monótona. Es la huella del desfase entre fecha de registro y fecha del hecho. En 2024 la Fiscalía por hecho queda 5.6% abajo, casi el doble del 2.75% que estimamos con el factor_corte. Parte del rezago puede ser mayor de lo que medimos con 31 días. 

### Circularidad

La correlación entre el núcleo SESNSP y nuestro C es 0.993, y la de rangos llega a 1.0 en 2024. En la CDMX, el SESNSP núcleo es C con otra fecha. R se estima de la encuesta, y si en el modelo de R metemos C como covariable, el coeficiente aprende algo real: si las alcaldías con más carpetas tienen más o menos visibilidad declarada. Eso no es una tautología.

#### Fay-Herriot

R̃_a = γ_a · R̂_a + (1 − γ_a) · x_a'β̂

Es decir, una mezcla de la estimación directa (lo que dice la encuesta) y de la predicción sintética (lo que dicen las covariables). El peso γ es menor cuanto más ruidosa es la directa, y tus d y k directos son muy ruidosos. Si log C está en x, entonces en las alcaldías con γ bajo:

R̃_a ≈ C_a^β y por tanto C_a / R̃_a ≈ C_a^(1−β)

Si β sale cerca de 1, el mapa corregido se aplana. La corrección cancela la señal de C y la geografía del mapa la decide el modelo, no los datos. Si β sale negativo, el mapa se estira. En ningún caso es neutral, y además depende justo de las alcaldías con menos muestra. A esto se suma que los artefactos de registro, como lo de Azcapotzalco en 2023 o el salto de “sin alcaldía”, entrarían por los dos lados.

Modelo	        Covariables de R	                            Papel
Principal	    Censo + agencias del MP + efectos fijos de año	El mapa corregido usa C una sola vez
Sensibilidad A	Principal + z_sesnsp_fuera_nucleo	
Sensibilidad B	Principal + z_sesnsp_nucleo	                    Versión original

Para cada modelo se reportan los γ por alcaldía, que dicen dónde manda la encuesta y dónde la covariable.

Sobre la sensibilidad A. Si queremos una covariable que varíe por año sin meter C, usaremos el SESNSP de los delitos fuera del núcleo: fraude, robo de vehículo, robo a negocio, narcomenudeo y los demás. Sigue viniendo del mismo registro, y eso es justamente lo útil. Mide la propensión general a registrar de cada alcaldía, que es un buen candidato para predecir la barrera institucional k, pero no comparte numerador con el mapa.

Sobre la variación temporal. Sin el SESNSP en el modelo principal, la variación entre años vendrá de los efectos aleatorios de área-tiempo del **Rao-Yu** y de los efectos fijos de año. Es exactamente para lo que existe el Rao-Yu, así que no pierde sentido.

In [ ]:
# Covariable SESNSP sin el numerador del mapa: delitos fuera del núcleo y fuera de violencia familiar

m = m.drop(columns=[c for c in m.columns if "fuera" in c])     # idempotente

m["S_fuera"] = m["S_total"] - m["S_nucleo"] - m["S_vf"]
assert (m["S_fuera"] > 0).all(), "Resta negativa: revisar núcleo o violencia familiar"
m["tasa18_sesnsp_fuera"] = 1e5 * m["S_fuera"] / m["pob_18_conapo"]
lf = np.log(m["tasa18_sesnsp_fuera"])
m["z_sesnsp_fuera"] = (lf - lf.mean()) / lf.std(ddof=0)

lc = np.log(m["tasa18_nucleo"])
print("Correlación con log C núcleo (80 alcaldía-año):")
for col in ["z_sesnsp_nucleo", "z_sesnsp_total", "z_sesnsp_fuera"]:
    print(f"  {col:<17} {np.corrcoef(lc, m[col])[0, 1]:.3f}")

# ¿Cuánto de la covariable es variación ENTRE alcaldías y cuánto ENTRE años?
# Si casi todo es entre alcaldías, aporta poca variación temporal al Rao-Yu.
for col in ["z_sesnsp_nucleo", "z_sesnsp_fuera"]:
    entre = m.groupby("cve_mun")[col].transform("mean")
    print(f"  {col}: {100 * entre.var() / m[col].var():.1f}% de su varianza es entre alcaldías")

m.to_parquet(CARPETA_SALIDA / "maestro_alcaldia_v1.parquet", index=False)
print(f"\nActualizado: maestro_alcaldia_v1.parquet · {m.shape[0]} × {m.shape[1]}")

Correlación con log C núcleo (80 alcaldía-año):
  z_sesnsp_nucleo   0.993
  z_sesnsp_total    0.910
  z_sesnsp_fuera    0.842
  z_sesnsp_nucleo: 92.2% de su varianza es entre alcaldías
  z_sesnsp_fuera: 97.1% de su varianza es entre alcaldías

Actualizado: maestro_alcaldia_v1.parquet · 80 × 37


- El SESNSP fuera del núcleo baja la correlación con C a 0.84. Está emparentado con el registro, como debe ser, pero ya no es el mismo numerador.
- El 97% de su varianza es entre alcaldías. Solo un 3% es variación de un año a otro dentro de la misma alcaldía; en el núcleo es 8%. O sea, el SESNSP no resuelve el problema temporal que te preocupaba: casi todo lo que aporta es un nivel fijo por alcaldía, que el censo ya cubre. Es otro argumento para dejarlo fuera del modelo principal y usarlo solo en las sensibilidades.

La variación temporal de R en el Rao-Yu proviene de los efectos aleatorios de área-tiempo y de efectos fijos de año. El SESNSP se descartó como covariable principal por dos razones: en su versión de núcleo es prácticamente la misma medición que el numerador del mapa (r = 0.993), y en cualquier versión aporta poca variación temporal (entre 3% y 8% de su varianza).

## Agencias del MP
### Servicios y sedes de la FGJ

In [ ]:
# ver qué trae

CARPETA_MP = RAIZ_REPO / "datos" / "crudos" / "Agencias del MP"
# En esa carpeta también está el de víctimas; nos quedamos con el que NO lo es
candidatos = [a for a in CARPETA_MP.glob("*.csv") if "victimas" not in a.name.lower()]
print("CSV de sedes:", [a.name for a in candidatos])
assert len(candidatos) == 1
RUTA_SEDES = candidatos[0]

for codif in ["utf-8-sig", "latin-1"]:
    try:
        sedes_raw = pd.read_csv(RUTA_SEDES, dtype=str, encoding=codif)
        break
    except UnicodeDecodeError:
        continue
print(f"Codificación: {codif} · {sedes_raw.shape[0]} filas × {sedes_raw.shape[1]} columnas\n")
print(sedes_raw.head(3).T)

# Columnas con pocos valores distintos: probablemente tipo de sede, alcaldía, etc.
print("\nColumnas categóricas (≤ 40 valores distintos):")
for col in sedes_raw.columns:
    n = sedes_raw[col].nunique()
    if n <= 40:
        print(f"\n--- {col} ({n} valores, {sedes_raw[col].isna().sum()} vacíos) ---")
        print(sedes_raw[col].value_counts(dropna=False).to_string())

CSV de sedes: ['e040c46f-8f49-4190-89d1-c08679ca2218.csv']
Codificación: utf-8-sig · 108 filas × 23 columnas

                                                                           0                                                  1                                                  2
id                                                                         0                                                  1                                                  2
tipo_de_servicio                                    Coordinación Territorial                           Coordinación Territorial                           Coordinación Territorial
sede                                                            AO-1  y AO-2                                        BJ-1 y BJ-2                                        BJ-3 y BJ-4
objeto_de_la_atencion      Las Coordinaciones territoriales buscan facili...  Las Coordinaciones territoriales buscan facili...  Las Coordinaciones territoriales buscan facil

- Coordinación Territorial (64) son las agencias del MP de barrio, donde la gente va a denunciar. Cada fila es una sede, y a veces agrupa dos agencias, como “AO-1 y AO-2”. Éstas son las que cuentan para la densidad y la distancia.
- Las otras 44 (fiscalías centrales y desconcentradas, atención a víctimas, juzgados familiares) atienden a toda la ciudad o a un tipo de delito, y la mayoría está en Cuauhtémoc. Por eso Cuauhtémoc tiene 36 filas en total. Contarlas por alcaldía inflaría Cuauhtémoc sin decir nada sobre el acceso de sus habitantes. Quedan fuera.

## Marco Geoestadístico

In [ ]:
# 2020: alcaldías y superficie

import geopandas as gpd
print(f"geopandas {gpd.__version__}")

CARPETA_MG = (RAIZ_REPO / "datos" / "crudos" / "Marco Geoestadístico"
              / "09_ciudaddemexico" / "conjunto_de_datos")
RUTA_MUN = CARPETA_MG / "09mun.shp"
assert RUTA_MUN.exists(), f"No encuentro {RUTA_MUN}"
# Un shapefile necesita sus archivos hermanos con el mismo nombre
for ext in [".shx", ".dbf", ".prj"]:
    assert RUTA_MUN.with_suffix(ext).exists(), f"Falta 09mun{ext}"

mun = gpd.read_file(RUTA_MUN)
print(f"{len(mun)} polígonos · columnas: {list(mun.columns)}")
print(f"Sistema de coordenadas original: {mun.crs}\n")
print(mun.drop(columns="geometry").head(16).to_string())

# --- Superficie --------------------------------------------------
# El marco viene en una proyección cónica de Lambert para todo México.
# Para medir áreas reproyectamos a UTM zona 14N (EPSG:32614), que cubre
# la CDMX con distorsión de escala menor a 0.1%.
mun_utm = mun.to_crs(epsg=32614)
mun["area_km2"] = mun_utm.geometry.area / 1e6
print(f"\nSuperficie total: {mun['area_km2'].sum():,.1f} km²")

geopandas 1.2.0
16 polígonos · columnas: ['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'NOMGEO', 'geometry']
Sistema de coordenadas original: PROJCS["MEXICO_ITRF_2008_LCC",GEOGCS["MEXICO_ITRF_2008",DATUM["International_Terrestrial_Reference_Frame_2008",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1061"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",12],PARAMETER["central_meridian",-102],PARAMETER["standard_parallel_1",17.5],PARAMETER["standard_parallel_2",29.5],PARAMETER["false_easting",2500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

   CVEGEO CVE_ENT CVE_MUN                  NOMGEO
0   09002      09     002            Azcapotzalco
1   09003      09     003                Coyoacán
2   09004      09     004   Cuajimalpa de Morelos
3   09005      09     005       Gustavo A. Madero
4   09006      09  

In [ ]:
# Densidad de población y sedes territoriales del MP

# Partimos del maestro guardado
m = pd.read_parquet(CARPETA_SALIDA / "maestro_alcaldia_v1.parquet")
mun = mun.rename(columns={"CVE_MUN": "cve_mun"})
assert set(mun["cve_mun"]) == set(claves["cve_mun"]), "Claves del marco ≠ catálogo"

# Idempotencia: quitamos lo que esta celda agrega
m = m.drop(columns=[c for c in m.columns if c in ("area_km2", "log_densidad")
                    or c.startswith(("n_sedes", "n_agencias", "sedes_mp", "agencias_mp"))])

# --- 1. Superficie y densidad (censo 2020, fija en el periodo) ---
m = m.merge(mun[["cve_mun", "area_km2"]], on="cve_mun", how="left", validate="many_to_one")
assert m["area_km2"].notna().all()
m["log_densidad"] = np.log(m["pob_total_censo2020"] / m["area_km2"])

# --- 2. Sedes territoriales → puntos en el mapa --------------------
terr = sedes_raw[sedes_raw["tipo_de_servicio"] == "Coordinación Territorial"].copy()
terr["lat"] = pd.to_numeric(terr["latitude"], errors="coerce")
terr["lon"] = pd.to_numeric(terr["longitude"], errors="coerce")
print(f"Sedes territoriales: {len(terr)} · sin coordenadas: {terr[['lat', 'lon']].isna().any(axis=1).sum()}")
assert terr[["lat", "lon"]].notna().all().all()

# EPSG:4326 = latitud/longitud en grados (como vienen en el archivo).
# Las pasamos a la proyección del marco para compararlas con los polígonos.
pts = gpd.GeoDataFrame(terr, geometry=gpd.points_from_xy(terr["lon"], terr["lat"]),
                       crs="EPSG:4326").to_crs(mun.crs)

# sjoin "within": a cada punto le pega la alcaldía cuyo polígono lo contiene
pts = gpd.sjoin(pts, mun[["cve_mun", "geometry"]], how="left", predicate="within")
pts = pts.drop(columns="index_right")
fuera = pts["cve_mun"].isna()
print(f"Sedes que no cayeron dentro de ningún polígono: {fuera.sum()}")
if fuera.any():
    # Un punto justo en la línea límite no está "dentro" de nadie: lo asignamos
    # a la alcaldía más cercana y lo mostramos para revisarlo.
    cerca = gpd.sjoin_nearest(pts.loc[fuera].drop(columns="cve_mun"),
                              mun[["cve_mun", "geometry"]], distance_col="dist_m")
    print(cerca[["sede", "alcaldia", "cve_mun", "dist_m"]].to_string())
    pts.loc[fuera, "cve_mun"] = cerca["cve_mun"].values

# --- 3. Control: ¿coincide con la alcaldía que dice el archivo? ---
POR_NOMBRE = dict(zip(claves["nombre_ascii"], claves["cve_mun"]))
POR_NOMBRE.update({"MAGDALENA CONTRERAS": "008", "CUAJIMALPA": "004"})   # variantes del archivo
pts["cve_por_nombre"] = pts["alcaldia"].map(lambda s: POR_NOMBRE.get(sin_acentos(s).strip()))
assert pts["cve_por_nombre"].notna().all(), "Hay un nombre de alcaldía sin traducir a clave"
distintas = pts[pts["cve_mun"] != pts["cve_por_nombre"]]
print(f"Sedes donde polígono y nombre no coinciden: {len(distintas)}")
if len(distintas):
    print(distintas[["sede", "alcaldia", "cve_por_nombre", "cve_mun", "lat", "lon"]].to_string())

# --- 4. Contar sedes y agencias por alcaldía ----------------------
# Una sede puede agrupar varias agencias ("AO-1 y AO-2"): contamos los códigos.
pts["n_agencias"] = (pts["sede"].str.upper().str.findall(r"[A-Z]+-\d+").str.len()
                     .clip(lower=1))
conteo = (pts.groupby("cve_mun").agg(n_sedes_mp=("sede", "size"), n_agencias_mp=("n_agencias", "sum"))
             .reindex(claves["cve_mun"], fill_value=0))
print(f"Total: {conteo['n_sedes_mp'].sum()} sedes · {conteo['n_agencias_mp'].sum()} agencias")

m = m.merge(conteo, left_on="cve_mun", right_index=True, how="left", validate="many_to_one")
m["sedes_mp_100k"]    = 1e5 * m["n_sedes_mp"]    / m["pob_18_censo2020"]
m["agencias_mp_100k"] = 1e5 * m["n_agencias_mp"] / m["pob_18_censo2020"]

print()
print(m[m["anio_ref"] == 2020][["nombre", "area_km2", "log_densidad", "n_sedes_mp", "n_agencias_mp",
                                "sedes_mp_100k", "agencias_mp_100k"]]
      .round(2).sort_values("agencias_mp_100k").to_string(index=False))

# --- 5. Guardar ---------------------------------------------------
assert m.shape[0] == 80 and not m.duplicated(["cve_mun", "anio_ref"]).any()
m.to_parquet(CARPETA_SALIDA / "maestro_alcaldia_v2.parquet", index=False)
pts.drop(columns="geometry").to_csv(CARPETA_SALIDA / "sedes_mp_territoriales.csv",
                                    index=False, encoding="utf-8-sig")
print(f"\nGuardado: maestro_alcaldia_v2.parquet · {m.shape[0]} × {m.shape[1]}")

Sedes territoriales: 64 · sin coordenadas: 0
Sedes que no cayeron dentro de ningún polígono: 0
Sedes donde polígono y nombre no coinciden: 0
Total: 64 sedes · 71 agencias

                nombre  area_km2  log_densidad  n_sedes_mp  n_agencias_mp  sedes_mp_100k  agencias_mp_100k
            Xochimilco    114.03          8.26           2              2           0.60              0.60
             Iztacalco     23.08          9.77           2              2           0.63              0.63
        Álvaro Obregón     95.82          8.98           3              4           0.51              0.68
               Tláhuac     85.78          8.43           2              2           0.69              0.69
            Iztapalapa    113.07          9.69           8             10           0.58              0.72
     Gustavo A. Madero     87.84          9.50           8              8           0.88              0.88
               Tlalpan    314.25          7.71           5              5      

- La densidad de agencias varía el triple, de 0.60 por 100 mil en Xochimilco a 1.85 en Milpa Alta. Es suficiente variación para que valga como covariable.
- Las dos más altas son engañosas, cada una a su manera. Cuauhtémoc (1.80) tiene muchas agencias para sus residentes, pero atienden también a la población flotante, que es el mismo problema de denominador que en las tasas. Milpa Alta (1.85) tiene pocas personas por agencia, pero en 298 km². Sus dos agencias quedan lejos de buena parte de la población.
- Iztacalco y Xochimilco tienen la menor densidad y apenas dos sedes cada una.

Esto refuerza que la densidad por habitante y la distancia miden cosas distintas, y que conviene tener las dos. Como el SESNSP ya quedó fuera del modelo principal, estas dos variables serán las únicas covariables institucionales.

## Principales resultados por AGEB y manzana urbana

In [ ]:
# Censo 2020 por AGEB y manzana: ver la estructura

RUTA_AGEB = RAIZ_REPO / "datos" / "crudos" / "AGEB" / "RESAGEBURB_09XLSX20.xlsx"
assert RUTA_AGEB.exists(), f"No encuentro {RUTA_AGEB}"

# Un Excel puede traer varias hojas; vemos cuáles hay
xls = pd.ExcelFile(RUTA_AGEB)
print("Hojas:", xls.sheet_names)

# header=None: no asumimos en qué renglón está el encabezado; vemos los primeros 8 tal cual
crudo = pd.read_excel(xls, sheet_name=xls.sheet_names[0], header=None, nrows=8, dtype=str)
print(crudo.iloc[:, :12].to_string())

Hojas: ['RESAGEBURB_09XLSX20']
        0                 1    2                                     3     4                             5     6    7        8        9        10      11
0  ENTIDAD           NOM_ENT  MUN                               NOM_MUN   LOC                       NOM_LOC  AGEB  MZA   POBTOT   POBFEM   POBMAS   P_0A2
1       09  Ciudad de México  000  Total de la entidad Ciudad de México  0000           Total de la entidad  0000  000  9209944  4805017  4404927  267151
2       09  Ciudad de México  002                          Azcapotzalco  0000           Total del municipio  0000  000   432205   227255   204950   11784
3       09  Ciudad de México  002                          Azcapotzalco  0001  Total de la localidad urbana  0000  000   432205   227255   204950   11784
4       09  Ciudad de México  002                          Azcapotzalco  0001             Total AGEB urbana  0010  000     3183     1695     1488      60
5       09  Ciudad de México  002            

In [ ]:
# Distancia media a la sede territorial del MP, ponderada por población (AGEB urbanas)

import time
inicio = time.time()

# --- 1. Población por AGEB (solo las columnas necesarias) ----------
pob_ageb = pd.read_excel(xls, sheet_name=xls.sheet_names[0], dtype=str,
                         usecols=["ENTIDAD", "MUN", "LOC", "NOM_LOC", "AGEB", "MZA", "POBTOT"])
print(f"Excel leído en {time.time() - inicio:.0f} s · {len(pob_ageb):,} filas")

pob_ageb = pob_ageb[(pob_ageb["NOM_LOC"] == "Total AGEB urbana") & (pob_ageb["MZA"] == "000")].copy()
pob_ageb["CVEGEO"] = pob_ageb["ENTIDAD"] + pob_ageb["MUN"] + pob_ageb["LOC"] + pob_ageb["AGEB"]
pob_ageb["pob"] = pd.to_numeric(pob_ageb["POBTOT"], errors="coerce")
print(f"Filas 'Total AGEB urbana': {len(pob_ageb):,} · claves repetidas: {pob_ageb['CVEGEO'].duplicated().sum()} "
      f"· población ilegible: {pob_ageb['pob'].isna().sum()}")
assert not pob_ageb["CVEGEO"].duplicated().any() and pob_ageb["pob"].notna().all()

# Cobertura: ¿qué parte de la población de cada alcaldía vive en AGEB urbanas?
cobertura = (pob_ageb.groupby("MUN")["pob"].sum()
             / censo.set_index("cve_mun")["POBTOT"] * 100).rename("cobertura_%")
print("\n% de la población de cada alcaldía en AGEB urbanas:")
print(cobertura.round(1).sort_values().to_string())

# --- 2. Polígonos de AGEB y pegado por clave -----------------------
ageb = gpd.read_file(CARPETA_MG / "09a.shp")
print(f"\nPolígonos 09a: {len(ageb):,} · columnas: {list(ageb.columns)}")
assert ageb["CVEGEO"].str.len().eq(13).all() and ageb["CVEGEO"].is_unique

ageb = ageb.merge(pob_ageb[["CVEGEO", "pob"]], on="CVEGEO", how="left",
                  validate="one_to_one", indicator=True)
print("Pegado polígono ↔ población:", ageb["_merge"].value_counts().to_dict())
sin_pob = set(pob_ageb["CVEGEO"]) - set(ageb["CVEGEO"])
print(f"AGEB con población pero sin polígono: {len(sin_pob)} "
      f"({pob_ageb.loc[pob_ageb['CVEGEO'].isin(sin_pob), 'pob'].sum():,} personas)")
# Un polígono sin fila de población es una AGEB deshabitada: pesa cero
ageb["pob"] = ageb["pob"].fillna(0)
ageb = ageb.drop(columns="_merge")

# --- 3. Distancia de cada AGEB a la sede más cercana ---------------
# En UTM 14N las distancias salen en metros. Usamos el centroide del
# polígono como punto representativo de dónde vive su población.
ageb_utm = ageb.to_crs(epsg=32614)
centros = ageb_utm[["CVEGEO", "CVE_MUN", "pob"]].copy()
centros = gpd.GeoDataFrame(centros, geometry=ageb_utm.geometry.centroid, crs=ageb_utm.crs)
sedes_utm = pts[["sede", "cve_mun", "geometry"]].rename(columns={"cve_mun": "cve_mun_sede"}).to_crs(epsg=32614)

cerca = gpd.sjoin_nearest(centros, sedes_utm, how="left", distance_col="dist_m")
cerca = cerca.drop_duplicates("CVEGEO")        # empates de distancia: una sola fila por AGEB
assert len(cerca) == len(centros) and cerca["dist_m"].notna().all()
print(f"\nAGEB cuya sede más cercana está en OTRA alcaldía: "
      f"{(cerca['CVE_MUN'] != cerca['cve_mun_sede']).mean() * 100:.1f}%")

# --- 4. Promedio por alcaldía, ponderado por población -------------
dist = (cerca.assign(pd_=cerca["pob"] * cerca["dist_m"])
             .groupby("CVE_MUN").agg(pd_=("pd_", "sum"), pob=("pob", "sum")))
dist["dist_mp_km"] = dist["pd_"] / dist["pob"] / 1000
dist = dist[["dist_mp_km"]].join(cobertura.rename("cobertura_ageb_pct"))
dist.index.name = "cve_mun"
assert set(dist.index) == set(claves["cve_mun"])

# --- 5. Al maestro (idempotente) -----------------------------------
m = m.drop(columns=[c for c in m.columns if c in ("dist_mp_km", "cobertura_ageb_pct")])
m = m.merge(dist, left_on="cve_mun", right_index=True, how="left", validate="many_to_one")
assert m["dist_mp_km"].notna().all()

print("\nDistancia media (línea recta) a la sede territorial más cercana:")
print(m[m["anio_ref"] == 2020][["nombre", "dist_mp_km", "cobertura_ageb_pct", "agencias_mp_100k"]]
      .round(2).sort_values("dist_mp_km").to_string(index=False))
print(f"\nCorrelación distancia vs. agencias por 100 mil: "
      f"{dist['dist_mp_km'].corr(m.groupby('cve_mun')['agencias_mp_100k'].first()):.2f}")

m.to_parquet(CARPETA_SALIDA / "maestro_alcaldia_v3.parquet", index=False)
print(f"\nGuardado: maestro_alcaldia_v3.parquet · {m.shape[0]} × {m.shape[1]}")

Excel leído en 200 s · 68,941 filas
Filas 'Total AGEB urbana': 2,433 · claves repetidas: 0 · población ilegible: 0

% de la población de cada alcaldía en AGEB urbanas:
MUN
009     84.3
013     97.1
004     97.7
012     97.9
011     98.2
008     99.5
005    100.0
002    100.0
006    100.0
007    100.0
003    100.0
010    100.0
014    100.0
015    100.0
016    100.0
017    100.0

Polígonos 09a: 2,431 · columnas: ['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'geometry']
Pegado polígono ↔ población: {'both': 2431, 'left_only': 0, 'right_only': 0}
AGEB con población pero sin polígono: 2 (7,108 personas)

AGEB cuya sede más cercana está en OTRA alcaldía: 16.9%

Distancia media (línea recta) a la sede territorial más cercana:
                nombre  dist_mp_km  cobertura_ageb_pct  agencias_mp_100k
            Cuauhtémoc        0.85              100.00              1.80
        Miguel Hidalgo        1.26              100.00              1.50
   Venustiano Carranza        1.27        

**Nota**: es distancia en línea recta, no tiempo de traslado. Subestima más donde la red vial es escasa o accidentada, como en las alcaldías del sur.

- 2,431 AGEB, cero sin empatar. Las 2 AGEB con población pero sin polígono suman 7,108 personas, el 0.08% de la ciudad. 
- La correlación de −0.35 confirma que densidad de agencias y distancia miden cosas distintas. Milpa Alta es el caso extremo: la alcaldía con más agencias por habitante (1.85) es también la que tiene más lejos su agencia (2.79 km).
- El 16.9% de las AGEB tiene más cerca una sede de otra alcaldía. 

El problema: Milpa Alta tiene solo 84.3% de cobertura, debajo del umbral de 85%. Le falta el 15.7% de su población, que vive en localidades rurales, justo la más alejada. Así que su 2.79 km subestima la distancia real. Como Milpa Alta es la alcaldía extrema, conviene arreglarlo.

In [ ]:
# Distancia con población urbana (AGEB) + rural (ITER)

import re

# --- 1. Localidades del ITER con coordenadas -----------------------
iter_loc = pd.read_csv(RUTA_ITER, dtype=str, encoding="utf-8-sig")
iter_loc = iter_loc[~iter_loc["LOC"].isin(["0000", "9998", "9999"])].copy()   # sin filas de total

# "99°01'37.123\" W" → grados + minutos/60 + segundos/3600; negativo si es Oeste o Sur
PATRON_DMS = re.compile(r"(\d+)\D+(\d+)\D+([\d.]+)\D*([NSEW])")
def dms_a_decimal(texto):
    g = PATRON_DMS.search(texto)
    valor = int(g[1]) + int(g[2]) / 60 + float(g[3]) / 3600
    return -valor if g[4] in "WS" else valor

iter_loc["lon"] = iter_loc["LONGITUD"].map(dms_a_decimal)
iter_loc["lat"] = iter_loc["LATITUD"].map(dms_a_decimal)
iter_loc["pob"] = pd.to_numeric(iter_loc["POBTOT"])

# --- 2. Rurales = localidades que NO están en el archivo de AGEB urbanas
urbanas = set(zip(pob_ageb["MUN"], pob_ageb["LOC"]))
es_rural = [(mu, lo) not in urbanas for mu, lo in zip(iter_loc["MUN"], iter_loc["LOC"])]
rurales = iter_loc[es_rural]
print(f"Localidades: {len(iter_loc)} · urbanas {len(iter_loc) - len(rurales)} · "
      f"rurales {len(rurales)} ({rurales['pob'].sum():,} personas)")

# --- 3. Distancia de cada localidad rural a la sede más cercana ----
pts_rur = gpd.GeoDataFrame(rurales[["MUN", "LOC", "NOM_LOC", "pob"]],
                           geometry=gpd.points_from_xy(rurales["lon"], rurales["lat"]),
                           crs="EPSG:4326").to_crs(epsg=32614)
cerca_rur = gpd.sjoin_nearest(pts_rur, sedes_utm, how="left", distance_col="dist_m")
cerca_rur = cerca_rur.drop_duplicates(["MUN", "LOC"])
assert len(cerca_rur) == len(pts_rur)

# --- 4. Juntar urbano + rural y recalcular -------------------------
todo = pd.concat([
    cerca[["CVE_MUN", "pob", "dist_m"]].rename(columns={"CVE_MUN": "cve_mun"}).assign(tipo="urbana"),
    cerca_rur[["MUN", "pob", "dist_m"]].rename(columns={"MUN": "cve_mun"}).assign(tipo="rural"),
], ignore_index=True)

def dist_ponderada(t):
    return (t["pob"] * t["dist_m"]).sum() / t["pob"].sum() / 1000

nueva = pd.DataFrame({
    "dist_mp_km_urb":     cerca.groupby("CVE_MUN").apply(dist_ponderada, include_groups=False),
    "dist_mp_km":         todo.groupby("cve_mun").apply(dist_ponderada, include_groups=False),
    "cobertura_total_pct": todo.groupby("cve_mun")["pob"].sum()
                           / censo.set_index("cve_mun")["POBTOT"] * 100,
    "pct_pob_rural":      todo[todo["tipo"] == "rural"].groupby("cve_mun")["pob"].sum()
                           / todo.groupby("cve_mun")["pob"].sum() * 100,
}).fillna({"pct_pob_rural": 0})
nueva.index.name = "cve_mun"

print("\nAntes (solo urbana) y después (urbana + rural):")
print(nueva.join(claves.set_index("cve_mun")["nombre"]).sort_values("dist_mp_km")
          .round(2).to_string())

# --- 5. Al maestro: la versión completa es la principal ------------
m = m.drop(columns=[c for c in m.columns if c.startswith(("dist_mp_km", "cobertura_"))])
m = m.merge(nueva[["dist_mp_km", "dist_mp_km_urb", "cobertura_total_pct"]],
            left_on="cve_mun", right_index=True, how="left", validate="many_to_one")
assert m["dist_mp_km"].notna().all() and len(m) == 80
m.to_parquet(CARPETA_SALIDA / "maestro_alcaldia_v3.parquet", index=False)
print(f"\nActualizado: maestro_alcaldia_v3.parquet · {m.shape[0]} × {m.shape[1]}")

Localidades: 634 · urbanas 35 · rurales 599 (64,312 personas)

Antes (solo urbana) y después (urbana + rural):
         dist_mp_km_urb  dist_mp_km  cobertura_total_pct  pct_pob_rural                  nombre
cve_mun                                                                                        
015                0.85        0.85               100.00           0.00              Cuauhtémoc
016                1.26        1.26               100.00           0.00          Miguel Hidalgo
017                1.27        1.27               100.00           0.00     Venustiano Carranza
014                1.37        1.37               100.00           0.00           Benito Juárez
003                1.37        1.37               100.00           0.00                Coyoacán
006                1.50        1.50               100.00           0.00               Iztacalco
002                1.63        1.63               100.00           0.00            Azcapotzalco
008                1.77  

Milpa Alta subió solo 0.08 km, de 2.79 a 2.87, aunque se agregó el 15.7% de su población. La razón es que su población rural vive en pueblos repartidos por toda la alcaldía (San Antonio Tecómitl, San Pablo Oztotepec, San Salvador Cuauhtenco), no aislada en un extremo. Esos pueblos quedan a distancias parecidas a las de las zonas urbanas de la propia Milpa Alta. Lo que la aleja de las agencias es la dispersión del poblamiento en general, no un grupo rural lejano.

Así que la versión solo urbana ya era una buena aproximación. Conservamos la completa como principal y la urbana como robustez. Ahora la decisión está respaldada por un dato y no por un supuesto. Las coberturas de Tláhuac y Tlalpan, de 99.2% y 99.4%, son casi seguro las 2 AGEB sin polígono.

Con esto ya están todas las covariables.

In [ ]:
# maestro final, diccionario, tabla de claves y revisión de colinealidad

# --- 1. Covariables estandarizadas (sobre las 16 alcaldías) -------
# Son fijas en el periodo: se estandarizan con una fila por alcaldía,
# para que ninguna alcaldía pese más por aparecer cinco veces.
COV_FIJAS = ["prop_fem_18", "prop_ocupada_12", "log_densidad", "eje_socio_1", "eje_socio_2",
             "agencias_mp_100k", "dist_mp_km"]
m = m.drop(columns=[f"z_{c}" for c in COV_FIJAS if f"z_{c}" in m.columns])
una = m.drop_duplicates("cve_mun").set_index("cve_mun")[COV_FIJAS]
z = (una - una.mean()) / una.std(ddof=0)
m = m.merge(z.add_prefix("z_"), left_on="cve_mun", right_index=True, how="left", validate="many_to_one")

# --- 2. Diccionario de columnas: descripción, fuente, ¿varía por año? ---
DESC = {
    "cvegeo": ("Clave geográfica INEGI (entidad + municipio)", "INEGI", False),
    "cve_mun": ("Clave de municipio INEGI, 3 dígitos. LLAVE de todos los pegados", "INEGI", False),
    "nombre": ("Nombre normalizado de la alcaldía", "Catálogo propio", False),
    "anio_ref": ("Año de referencia = año del hecho = año de la ENVIPE − 1", "—", True),
    "pob_total_conapo": ("Población total a mitad de año", "CONAPO 1990–2040", True),
    "pob_18_conapo": ("Población de 18+; el tramo 18–19 se estima con la fracción del censo", "CONAPO + Censo 2020", True),
    "pob_18_fem_conapo": ("Mujeres de 18+ (mismo método)", "CONAPO + Censo 2020", True),
    "pob_total_censo2020": ("Población total", "Censo 2020, ITER", False),
    "pob_18_censo2020": ("Población de 18+", "Censo 2020, ITER", False),
    "prop_fem_18": ("Mujeres de 18+ / población de 18+", "Censo 2020, ITER", False),
    "prop_ocupada_12": ("Población ocupada / población de 12+", "Censo 2020, ITER", False),
    "eje_socio_1": ("PCA, eje 1 (74.3%). Positivo = más rezago", "Censo 2020, ITER", False),
    "eje_socio_2": ("PCA, eje 2 (12.3%). Positivo = más desocupación", "Censo 2020, ITER", False),
    "factor_corte_nucleo": ("Factor por rezago del corte (31-ene-2025); ≠ 1 solo en 2024. NO aplicado a C", "FGJ", True),
    "factor_sin_alc_nucleo": ("Factor de prorrateo de carpetas sin alcaldía. NO aplicado a C", "FGJ", True),
    "S_total": ("Carpetas del fuero común, por mes de REGISTRO", "SESNSP 2015–2025", True),
    "S_nucleo": ("Carpetas SESNSP en subtipos equivalentes al núcleo", "SESNSP 2015–2025", True),
    "S_vf": ("Carpetas SESNSP de violencia familiar", "SESNSP 2015–2025", True),
    "S_fuera": ("SESNSP total − núcleo − violencia familiar", "SESNSP 2015–2025", True),
    "area_km2": ("Superficie (proyección UTM 14N)", "Marco Geoestadístico 2020", False),
    "log_densidad": ("log(población censo 2020 / km²)", "Censo 2020 + Marco Geoestadístico", False),
    "n_sedes_mp": ("Sedes territoriales del MP (Coordinaciones Territoriales)", "FGJ, Servicios y sedes (lista 2026)", False),
    "n_agencias_mp": ("Agencias territoriales (códigos dentro de cada sede)", "FGJ, Servicios y sedes (lista 2026)", False),
    "sedes_mp_100k": ("Sedes por 100 mil habitantes de 18+ (censo)", "FGJ + Censo 2020", False),
    "agencias_mp_100k": ("Agencias por 100 mil habitantes de 18+ (censo)", "FGJ + Censo 2020", False),
    "dist_mp_km": ("Distancia en línea recta a la sede más cercana, ponderada por población (AGEB urbanas + localidades rurales)",
                   "FGJ + Censo (AGEB e ITER) + Marco Geoestadístico", False),
    "dist_mp_km_urb": ("Igual que dist_mp_km, solo AGEB urbanas (robustez)", "FGJ + Censo AGEB + Marco Geoestadístico", False),
    "cobertura_total_pct": ("% de la población censal incluida en el cálculo de la distancia", "Censo 2020", False),
}
VERSIONES_C = {"nucleo": "núcleo: BPCOD 04, 05, 10, 11, 13, 14 (alta + dudosas aceptadas). PRINCIPAL",
               "nucleo_alta": "núcleo, solo confianza alta", "nucleo_vf": "núcleo + violencia familiar",
               "amplio": "todo lo asignado a un BPCOD salvo 03 y 15"}
for v, txt in VERSIONES_C.items():
    DESC[f"C_{v}"] = (f"Carpetas por alcaldía y año del HECHO — {txt}", "FGJ datos abiertos (corte 31-ene-2025)", True)
    DESC[f"tasa18_{v}"] = (f"C_{v} por 100 mil habitantes de 18+ (CONAPO)", "FGJ + CONAPO", True)
    DESC[f"tasa_tot_{v}"] = (f"C_{v} por 100 mil habitantes totales (CONAPO)", "FGJ + CONAPO", True)
for s in ["total", "nucleo", "fuera"]:
    DESC[f"tasa18_sesnsp_{s}"] = (f"S_{s} por 100 mil habitantes de 18+ (CONAPO)", "SESNSP + CONAPO", True)
    DESC[f"z_sesnsp_{s}"] = (f"log(tasa18_sesnsp_{s}) estandarizado sobre las 80 filas", "SESNSP + CONAPO", True)
for c in COV_FIJAS:
    DESC[f"z_{c}"] = (f"{c} estandarizada sobre las 16 alcaldías", DESC[c][1], False)

sin_doc = set(m.columns) - set(DESC)
assert not sin_doc, f"Columnas sin documentar: {sin_doc}"

# --- 3. Verificación: ¿la marca "varía por año" coincide con los datos?
varia_real = m.groupby("cve_mun").nunique().gt(1).any()
diccionario = pd.DataFrame([(c, *DESC[c], bool(varia_real[c]) if c in varia_real else False)
                            for c in m.columns],
                           columns=["columna", "descripcion", "fuente", "varia_por_anio", "varia_en_datos"])
mal = diccionario[diccionario["varia_por_anio"] != diccionario["varia_en_datos"]]
assert mal.empty, f"La marca no coincide con los datos:\n{mal}"
print("Columnas que SÍ cambian por año:")
print("  ", [c for c in diccionario.loc[diccionario["varia_por_anio"], "columna"] if c != "anio_ref"])

# --- 4. Colinealidad entre covariables candidatas (16 alcaldías) ---
cand = m.drop_duplicates("cve_mun").set_index("nombre")[
    [f"z_{c}" for c in COV_FIJAS] + ["z_sesnsp_fuera"]]
corr = cand.corr()
print("\nCorrelaciones entre covariables candidatas (16 alcaldías):")
print(corr.round(2).to_string())
altas = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr) for b in corr.columns[i + 1:]
         if abs(corr.loc[a, b]) >= 0.7]
print("\nPares con |r| ≥ 0.7 (no conviene meterlos juntos al modelo):")
for a, b, r in sorted(altas, key=lambda t: -abs(t[2])):
    print(f"  {a:<22} {b:<22} {r:+.2f}")

# --- 5. Tabla de claves final (entregable 2) ------------------------
otros = pd.DataFrame({
    "nombre_marco_geo": mun.set_index("cve_mun")["NOMGEO"],
    "nombre_sesnsp": sesnsp[sesnsp["cve_mun"].isin(claves["cve_mun"])].drop_duplicates("cve_mun")
                     .set_index("cve_mun")["Municipio"],
    "nombre_sedes_fgj": pts.drop_duplicates("cve_mun").set_index("cve_mun")["alcaldia"],
})
claves_final = claves.drop(columns=[c for c in otros.columns if c in claves.columns]) \
                     .merge(otros, left_on="cve_mun", right_index=True, how="left", validate="one_to_one")
assert len(claves_final) == 16 and claves_final.notna().all().all()
claves_final.to_csv(CARPETA_SALIDA / "claves_alcaldias.csv", index=False, encoding="utf-8-sig")

# --- 6. Guardar el maestro final y su diccionario ------------------
assert len(m) == 80 and not m.duplicated(["cve_mun", "anio_ref"]).any()
print(f"\nValores faltantes en el maestro: {m.isna().sum().sum()}")
assert m.isna().sum().sum() == 0
m.to_parquet(CARPETA_SALIDA / "maestro_alcaldia.parquet", index=False)
diccionario.drop(columns="varia_en_datos").to_csv(CARPETA_SALIDA / "diccionario_maestro.csv",
                                                  index=False, encoding="utf-8-sig")
print(f"Guardado: maestro_alcaldia.parquet ({m.shape[0]} × {m.shape[1]}) · "
      f"diccionario_maestro.csv · claves_alcaldias.csv ({claves_final.shape[1]} columnas)")

Columnas que SÍ cambian por año:
   ['C_nucleo', 'C_nucleo_alta', 'C_nucleo_vf', 'C_amplio', 'pob_total_conapo', 'pob_18_conapo', 'pob_18_fem_conapo', 'tasa18_nucleo', 'tasa_tot_nucleo', 'tasa18_nucleo_alta', 'tasa_tot_nucleo_alta', 'tasa18_nucleo_vf', 'tasa_tot_nucleo_vf', 'tasa18_amplio', 'tasa_tot_amplio', 'factor_corte_nucleo', 'factor_sin_alc_nucleo', 'S_total', 'S_nucleo', 'S_vf', 'tasa18_sesnsp_total', 'z_sesnsp_total', 'tasa18_sesnsp_nucleo', 'z_sesnsp_nucleo', 'S_fuera', 'tasa18_sesnsp_fuera', 'z_sesnsp_fuera']

Correlaciones entre covariables candidatas (16 alcaldías):
                    z_prop_fem_18  z_prop_ocupada_12  z_log_densidad  z_eje_socio_1  z_eje_socio_2  z_agencias_mp_100k  z_dist_mp_km  z_sesnsp_fuera
z_prop_fem_18                1.00               0.25            0.52          -0.90          -0.02                0.17         -0.60            0.58
z_prop_ocupada_12            0.25               1.00           -0.01          -0.38          -0.55                0.